<a href="https://colab.research.google.com/github/ysy3046/ai-data-bootcamp/blob/main/project_3_data_%EB%AA%A8%ED%98%95_%EA%B2%80%EC%A6%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [72]:
!wget -q -O anchors.json "https://drive.google.com/uc?export=download&id=17CX9p8e6aEWp9WmksoHOtZxXzy_2ANSN"

In [73]:
!wget -q -O heritage_visitors_daily.parquet "https://drive.google.com/uc?export=download&id=1BP3zwSCFIJ5wmF5RBI-vbWBspKb9HBqI"

In [74]:
!wget -q -O sites.csv "https://drive.google.com/uc?export=download&id=10TRYm-vijCSw1If3njEf-h9i-n3KBTkP"

In [75]:
!wget -q -O weather_daily.parquet "https://drive.google.com/uc?export=download&id=1H6RmByTZH63-TGNk2Mxr6Zest-dbkB45"

In [76]:
	!wget -q -O calendar_kr_2015_2025.csv "https://drive.google.com/uc?export=download&id=1lDtkykv-Be0WEIGX4ts5Fp9MlIYULtPo"

In [77]:
!ls -lh *.parquet *.csv *.json

-rw-r--r-- 1 root root  23K Oct 12 13:02 anchors.json
-rw-r--r-- 1 root root  91K Oct 12 13:02 calendar_kr_2015_2025.csv
-rw-r--r-- 1 root root 716K Oct 12 13:02 heritage_visitors_daily.parquet
-rw-r--r-- 1 root root 8.6K Oct 12 13:02 sites.csv
-rw-r--r-- 1 root root 455K Oct 12 13:02 weather_daily.parquet


In [78]:
import pandas as pd
import json

visits = pd.read_parquet("heritage_visitors_daily.parquet")
sites = pd.read_csv("sites.csv")
weather = pd.read_parquet("weather_daily.parquet")
calendar = pd.read_csv("calendar_kr_2015_2025.csv")

with open("anchors.json", encoding="utf-8") as f:
    anchors = json.load(f)

visits.head()

,date,group,site,paid,free,domestic,foreign_en,foreign_ja,foreign_zh,foreign_etc,total
0,2015-01-01,궁,경복궁,13281,2363,10834,207,199,3907,497,15644
1,2015-01-01,궁,창덕궁,3253,847,2613,932,336,204,15,4100
2,2015-01-01,궁,창경궁,709,305,960,29,16,9,0,1014
3,2015-01-01,궁,덕수궁,1620,729,1853,98,32,344,22,2349
4,2015-01-02,궁,경복궁,10676,2028,8654,125,84,3718,123,12704


In [79]:
foreign_cols = ["foreign_en", "foreign_ja", "foreign_zh", "foreign_etc"]

assert len(visits) == anchors["rows"]
assert visits["site"].nunique() == anchors["n_sites"]
assert visits["total"].sum() == anchors["sums"]["total"]
assert visits[foreign_cols].sum().sum() == anchors["sums"]["foreign_all"]
assert (visits["total"] == 0).sum() == anchors["zero_total_site_days"]["all"]
assert len(weather) == anchors["weather"]["rows"]
assert len(calendar) == anchors["calendar"]["rows"]

print("검산 7개 모두 통과!")

검산 7개 모두 통과!


In [80]:
print("열별 결측 개수")
print(visits.isna().sum())

print("\n열별 자료형")
print(visits.dtypes)

print("\n날짜 × 개소 중복 개수")
print(visits.duplicated(["date", "site"]).sum())

열별 결측 개수
date           0
group          0
site           0
paid           0
free           0
domestic       1
foreign_en     0
foreign_ja     0
foreign_zh     0
foreign_etc    0
total          0
dtype: int64

열별 자료형
date           datetime64[ns]
group                  object
site                   object
paid                    int64
free                    int64
domestic                Int64
foreign_en              int64
foreign_ja              int64
foreign_zh              int64
foreign_etc             int64
total                   int64
dtype: object

날짜 × 개소 중복 개수
0


In [81]:
# 결측이 있는 행
display(visits[visits.isna().any(axis=1)])

# 관람객 관련 열에 음수가 있는 행
count_cols = ["paid", "free", "domestic", *foreign_cols, "total"]
display(visits[(visits[count_cols] < 0).any(axis=1)])

,date,group,site,paid,free,domestic,foreign_en,foreign_ja,foreign_zh,foreign_etc,total
25732,2021-08-15,궁,경복궁,0,0,<NA>,0,0,0,0,0


,date,group,site,paid,free,domestic,foreign_en,foreign_ja,foreign_zh,foreign_etc,total
4926,2018-05-16,궁,창경궁,-241,604,259,66,8,30,0,363
10618,2019-08-06,궁,경복궁,-1,12,11,0,0,0,0,11
26216,2021-09-06,궁,경복궁,712,513,1094,115,-1,3,14,1225


In [82]:
foreign_total = visits[foreign_cols].sum(axis=1)
known = visits["domestic"].notna()

mismatch = visits.loc[
    known & (
        visits["total"] != visits["domestic"] + foreign_total
    )
]

print("합계 불일치 행 수:", len(mismatch))
display(
    mismatch.groupby([mismatch["date"].dt.year, "site"])
    .size()
    .rename("불일치 행 수")
)

합계 불일치 행 수: 3436


date  site  
2022  김포장릉      313
      서삼릉       312
      서오릉       313
      선릉        310
      세종대왕유적    313
      양주온릉      311
      영휘원       313
      융릉        313
      의릉        313
      파주삼릉      312
      파주장릉      313
Name: 불일치 행 수, dtype: int64

In [83]:
for site, data in visits.groupby("site"):
    expected = pd.date_range(data["date"].min(), data["date"].max())
    missing = expected.difference(pd.DatetimeIndex(data["date"]))

    if len(missing):
        print(
            f"{site}: {len(missing)}일 누락 "
            f"({missing.min():%Y-%m-%d} ~ {missing.max():%Y-%m-%d})"
        )

경복궁: 53일 누락 (2020-05-30 ~ 2020-07-21)
광릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
김포장릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
덕수궁: 53일 누락 (2020-05-30 ~ 2020-07-21)
동구릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
사릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
서삼릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
서오릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
선릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
세종대왕유적: 53일 누락 (2020-05-30 ~ 2020-07-21)
양주온릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
영휘원: 53일 누락 (2020-05-30 ~ 2020-07-21)
융릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
의릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
정릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
창경궁: 53일 누락 (2020-05-30 ~ 2020-07-21)
창덕궁: 53일 누락 (2020-05-30 ~ 2020-07-21)
태릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
파주삼릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
파주장릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
헌릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
홍유릉: 53일 누락 (2020-05-30 ~ 2020-07-21)


In [84]:
check = visits.merge(
    sites[["site", "closed_weekday"]],
    on="site",
    how="left",
    validate="many_to_one"
)

weekday_map = {0: "월", 1: "화", 2: "수", 3: "목",
               4: "금", 5: "토", 6: "일"}
check["weekday"] = check["date"].dt.dayofweek.map(weekday_map)

regular_closed = check["weekday"] == check["closed_weekday"]

print("정기 휴관 요일인데 관람객이 있는 행:",
      (regular_closed & check["total"].gt(0)).sum())

print("정기 휴관 요일이 아닌데 0명인 행:",
      (~regular_closed & check["total"].eq(0)).sum())

정기 휴관 요일인데 관람객이 있는 행: 918
정기 휴관 요일이 아닌데 0명인 행: 903


In [85]:
calendar["date"] = pd.to_datetime(calendar["date"])

check = check.merge(
    calendar[["date", "is_holiday"]],
    on="date",
    how="left",
    validate="many_to_one"
)

exceptions = check[
    check["weekday"].eq(check["closed_weekday"])
    & check["total"].gt(0)
]

print(exceptions.groupby("is_holiday").size())

is_holiday
0    277
1    641
dtype: int64


In [86]:
# 원본은 유지하고, 첫 실험용 데이터만 따로 만들기
model_data = visits.loc[
    visits["date"] >= "2023-01-01"
].copy()

model_data = model_data.sort_values(
    ["site", "date"]
).reset_index(drop=True)

print("분석 기간:",
      model_data["date"].min(),
      "~", model_data["date"].max())
print("개소 수:", model_data["site"].nunique())
print("행 수:", len(model_data))


분석 기간: 2023-01-01 00:00:00 ~ 2025-12-31 00:00:00
개소 수: 23
행 수: 25208


In [87]:
special_days = calendar.loc[
    calendar["date"].dt.year.eq(2025)
    & (
        calendar["is_holiday"].eq(1)
        | calendar["is_last_wednesday"].eq(1)
    ),
    ["date", "weekday", "holiday_name", "is_last_wednesday"]
]

display(special_days)

,date,weekday,holiday_name,is_last_wednesday
3653,2025-01-01,수,신정연휴,0
3679,2025-01-27,월,임시공휴일,0
3680,2025-01-28,화,설날 전날,0
3681,2025-01-29,수,설날,1
3682,2025-01-30,목,설날 다음날,0
3709,2025-02-26,수,NaN,1
3712,2025-03-01,토,삼일절,0
3714,2025-03-03,월,삼일절 대체 휴일,0
3737,2025-03-26,수,NaN,1
3772,2025-04-30,수,NaN,1


In [88]:
HORIZON = 7

# 예측 기간 7일이 모두 2025년 안에 들어오는 원점
origins = pd.date_range(
    start="2025-01-01",
    end="2025-12-25",
    freq="7D"
)

print("원점 수:", len(origins))
print("첫 원점:", origins.min())
print("마지막 원점:", origins.max())

원점 수: 52
첫 원점: 2025-01-01 00:00:00
마지막 원점: 2025-12-24 00:00:00


In [89]:
zero_runs = []

for site, data in visits.groupby("site"):
    data = data.sort_values("date").copy()

    # 날짜가 끊기거나, 0명 여부가 바뀌면 새 구간
    new_run = (
        data["date"].diff().ne(pd.Timedelta(days=1))
        | data["total"].eq(0).ne(data["total"].shift().eq(0))
    )

    for _, run in data.groupby(new_run.cumsum()):
        if run["total"].eq(0).all() and len(run) >= 5:
            if run["date"].max() >= pd.Timestamp("2022-01-01"):
                zero_runs.append({
                    "site": site,
                    "시작": run["date"].min(),
                    "끝": run["date"].max(),
                    "연속 일수": len(run)
                })

zero_runs_df = pd.DataFrame(zero_runs)
display(zero_runs_df)

,site,시작,끝,연속 일수
0,광릉,2024-11-27,2024-12-02,6
1,광릉,2025-07-20,2025-07-25,6
2,김포장릉,2024-11-27,2024-12-02,6
3,동구릉,2024-11-28,2024-12-02,5
4,사릉,2024-11-27,2024-12-02,6
5,서삼릉,2024-11-27,2024-12-02,6
6,서오릉,2024-11-27,2024-12-02,6
7,선릉,2024-11-27,2024-12-06,10
8,세종대왕유적,2024-11-28,2024-12-02,5
9,양주온릉,2024-11-27,2024-12-02,6


In [90]:
# 팀 기준에 맞춰 2022년부터 점검용 데이터 만들기
audit_data = visits.loc[
    visits["date"] >= "2022-01-01"
].copy()

# 행 순서가 아니라 정확히 7일 전 날짜로 연결
lag7 = audit_data[["date", "site", "total"]].copy()
lag7["date"] += pd.Timedelta(days=7)
lag7 = lag7.rename(columns={"total": "total_7days_ago"})

pairs = audit_data.merge(
    lag7, on=["date", "site"], how="left",
    validate="one_to_one"
)

valid = pairs[
    pairs["total"].gt(0)
    & pairs["total_7days_ago"].gt(0)
].copy()

valid["weekly_diff"] = (
    valid["total"] - valid["total_7days_ago"]
).abs()

display(
    valid.groupby("site")["weekly_diff"]
    .agg(유효쌍="count", 스케일="mean")
)

,유효쌍,스케일
site,,
경복궁,1249,7291.822258
광릉,1193,128.697402
김포장릉,1203,317.384871
덕수궁,1232,3385.648539
동구릉,1205,284.034025
사릉,1201,30.790175
서삼릉,1201,66.348043
서오릉,1200,467.304167
선릉,1198,617.858097


In [91]:
first_origin_scale = (
    valid.loc[valid["date"] < "2025-01-01"]
    .groupby("site")["weekly_diff"]
    .agg(유효쌍="count", 스케일="mean")
    .reindex(sites["site"])
)

display(first_origin_scale)

print(
    "분모 확인이 필요한 개소:",
    first_origin_scale.index[
        first_origin_scale["스케일"].isna()
        | first_origin_scale["스케일"].le(0)
    ].tolist()
)

,유효쌍,스케일
site,,
경복궁,943,6604.765642
창덕궁,903,1917.478405
창경궁,896,1734.959821
덕수궁,924,2992.712121
종묘,596,639.823826
동구릉,900,246.260000
광릉,898,121.688196
홍유릉,899,160.687430
사릉,896,30.389509


분모 확인이 필요한 개소: []


In [92]:
scale_checks = []

for origin in origins:
    scales = (
        valid.loc[valid["date"] < origin]
        .groupby("site")["weekly_diff"]
        .mean()
        .reindex(sites["site"])
    )

    for site, scale in scales.items():
        scale_checks.append({
            "origin": origin,
            "site": site,
            "scale": scale
        })

scale_checks_df = pd.DataFrame(scale_checks)

bad_scales = scale_checks_df[
    scale_checks_df["scale"].isna()
    | scale_checks_df["scale"].le(0)
]

print("확인한 개소 × 원점:", len(scale_checks_df))
print("분모 문제 건수:", len(bad_scales))

확인한 개소 × 원점: 1196
분모 문제 건수: 0


In [93]:
weather["date"] = pd.to_datetime(weather["date"])
calendar["date"] = pd.to_datetime(calendar["date"])

merged_check = audit_data.merge(
    weather,
    on=["date", "site"],
    how="left",
    validate="one_to_one",
    indicator="weather_match"
).merge(
    calendar,
    on="date",
    how="left",
    validate="many_to_one",
    indicator="calendar_match"
)

print("병합 전후 행 수:", len(audit_data), len(merged_check))
print("날씨 연결 실패:", merged_check["weather_match"].eq("left_only").sum())
print("달력 연결 실패:", merged_check["calendar_match"].eq("left_only").sum())

병합 전후 행 수: 33238 33238
날씨 연결 실패: 0
달력 연결 실패: 0


In [94]:
work_data = visits.loc[
    visits["date"] >= "2023-01-01"
].copy()

work_data = work_data.merge(
    sites[["site", "closed_weekday"]],
    on="site",
    how="left",
    validate="many_to_one"
).merge(
    calendar,
    on="date",
    how="left",
    validate="many_to_one"
)

# 점검용 표시: 실제 개방·휴관 확정값은 아님
work_data["is_regular_closed_day"] = (
    work_data["weekday"] == work_data["closed_weekday"]
)
work_data["is_zero_record"] = work_data["total"].eq(0)

work_data = work_data.sort_values(
    ["site", "date"]
).reset_index(drop=True)

print("행 수:", len(work_data))
print("개소 수:", work_data["site"].nunique())
print("달력 연결 실패:", work_data["weekday"].isna().sum())

행 수: 25208
개소 수: 23
달력 연결 실패: 0


In [95]:
HORIZON = 7  # 팀 초안 기준, 확정 전

origins = pd.date_range(
    start="2025-07-01",
    periods=26,
    freq="7D"
)

print("원점 수:", len(origins))
print("첫 원점:", origins.min())
print("마지막 원점:", origins.max())
print("마지막 예측일:",
      origins.max() + pd.Timedelta(days=HORIZON - 1))

원점 수: 26
첫 원점: 2025-07-01 00:00:00
마지막 원점: 2025-12-23 00:00:00
마지막 예측일: 2025-12-29 00:00:00


In [96]:
# 원본 total은 보존
work_data["target_for_model"] = work_data["total"].astype(float)
work_data.loc[
    work_data["is_zero_record"], "target_for_model"
] = float("nan")

print("학습용 값이 NaN인 행:",
      work_data["target_for_model"].isna().sum())

print("정기 휴관 요일인데 관람객이 있는 행:",
      (
          work_data["is_regular_closed_day"]
          & ~work_data["is_zero_record"]
      ).sum())

학습용 값이 NaN인 행: 3700
정기 휴관 요일인데 관람객이 있는 행: 325


In [115]:
lag7 = work_data[["date", "site", "target_for_model"]].copy()
lag7["date"] += pd.Timedelta(days=7)
lag7 = lag7.rename(columns={"target_for_model": "target_lag7"})

scale_data = work_data.merge(
    lag7, on=["date", "site"],
    how="left", validate="one_to_one"
)

scale_data["weekly_diff"] = (
    scale_data["target_for_model"] - scale_data["target_lag7"]
).abs()

scale_table = pd.concat([
    scale_data.loc[scale_data["date"] < origin]
    .groupby("site")["weekly_diff"].mean()
    .reindex(sites["site"])
    .rename("scale")
    .reset_index()
    .assign(origin=origin)
    for origin in origins
], ignore_index=True)

print("개소 × 원점:", len(scale_table))
print("분모 문제:",
      (scale_table["scale"].isna() | scale_table["scale"].le(0)).sum())

개소 × 원점: 1196
분모 문제: 0


In [116]:
baseline_rows = []

for origin in origins:
    for site, data in work_data.groupby("site"):
        history = data.loc[data["date"] < origin]
        opened = history.dropna(subset=["target_for_model"])

        if opened.empty:
            raise ValueError(f"{site}: 사용 가능한 과거 기록 없음")

        naive = opened.iloc[-1]["target_for_model"]

        recent = opened.loc[
            opened["date"] >= origin - pd.Timedelta(days=7)
        ]
        # 직전 7일에 유효 기록이 없으면 임의로 대체하지 않고 표시
        mean7 = recent["target_for_model"].mean()

        for date in pd.date_range(origin, periods=HORIZON):
            same_weekday = opened.loc[
                opened["date"].dt.dayofweek == date.dayofweek
            ]
            seasonal = (
                same_weekday.iloc[-1]["target_for_model"]
                if not same_weekday.empty else float("nan")
            )

            baseline_rows.append({
                "origin": origin,
                "site": site,
                "date": date,
                "naive": naive,
                "seasonal_naive": seasonal,
                "mean7": mean7
            })

baseline_predictions = pd.DataFrame(baseline_rows)

print("予測 행 수:", len(baseline_predictions))
print("예측값 결측:")
print(baseline_predictions[
    ["naive", "seasonal_naive", "mean7"]
].isna().sum())
display(baseline_predictions.head())

予測 행 수: 8372
예측값 결측:
naive             0
seasonal_naive    0
mean7             0
dtype: int64


,origin,site,date,naive,seasonal_naive,mean7
0,2025-01-01,경복궁,2025-01-01,17300.0,34364.0,18422.166667
1,2025-01-01,경복궁,2025-01-02,17300.0,13284.0,18422.166667
2,2025-01-01,경복궁,2025-01-03,17300.0,13348.0,18422.166667
3,2025-01-01,경복궁,2025-01-04,17300.0,15426.0,18422.166667
4,2025-01-01,경복궁,2025-01-05,17300.0,16811.0,18422.166667


In [120]:
evaluation = baseline_predictions.merge(
    work_data[["date", "site", "total"]],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left",
    validate="many_to_one"
)

# 임시 평가 대상: 실제 관람객이 있는 날
evaluation = evaluation.loc[evaluation["total"].gt(0)].copy()

mase_results = []

for method in ["naive", "seasonal_naive", "mean7"]:
    evaluation[f"ae_{method}"] = (
        evaluation["total"] - evaluation[method]
    ).abs()

    by_pair = (
        evaluation.groupby(["site", "origin"])[f"ae_{method}"]
        .mean()
        .rename("MAE")
        .reset_index()
        .merge(
            scale_table,
            on=["site", "origin"],
            validate="one_to_one"
        )
    )

    by_pair["MASE"] = by_pair["MAE"] / by_pair["scale"]
    by_pair["method"] = method
    mase_results.append(by_pair)

mase_by_pair = pd.concat(mase_results, ignore_index=True)

display(
    mase_by_pair.groupby("method")
    .agg(MASE=("MASE", "mean"), 평가쌍=("MASE", "count"))
    .sort_values("MASE")
)

,MASE,평가쌍
method,,
naive,1.204833,1196
mean7,1.218304,1196
seasonal_naive,1.244695,1196


In [118]:

mase_by_origin = mase_by_pair.pivot_table(
    index="origin",
    columns="method",
    values="MASE",
    aggfunc="mean"
)

display(mase_by_origin)

method,mean7,naive,seasonal_naive
origin,,,
2025-01-01,1.418225,1.197469,1.322063
2025-01-08,0.634281,0.414627,0.733128
2025-01-15,0.492318,0.534912,0.395610
2025-01-22,1.183566,1.323161,1.163489
2025-01-29,1.313600,1.409541,2.089740
2025-02-05,1.690303,0.424149,1.588596
2025-02-12,0.673552,0.629491,0.634249
2025-02-19,0.491527,0.476646,0.398746
2025-02-26,0.755900,0.888662,0.825386


In [101]:
holiday_check = evaluation.loc[
    evaluation["origin"].isin(
        pd.to_datetime(["2025-10-07", "2025-10-14"])
    )
].copy()

# 양수: 실제보다 많이 예측 / 음수: 적게 예측
holiday_check["error_mean7"] = (
    holiday_check["mean7"] - holiday_check["total"]
)

display(
    holiday_check.groupby("origin").agg(
        평가건수=("total", "size"),
        과소예측건수=("error_mean7", lambda x: (x < 0).sum()),
        과대예측건수=("error_mean7", lambda x: (x > 0).sum()),
        평균절대오차=("ae_mean7", "mean")
    )
)

,평가건수,과소예측건수,과대예측건수,평균절대오차
origin,,,,
2025-10-07,122,92,30,6158.395589
2025-10-14,138,6,132,4098.102484


서영, 직전 7일 평균이 연휴 주에는 주로 적게, 다음 주에는 주로 많이 예측한 패턴이 확인됐어.
- 10/7 원점: 122건 중 92건 과소예측(75.4%)
- 10/14 원점: 138건 중 132건 과대예측(95.7%)
이건 개소×날짜 건수 기준이야. 명절이 원인이라고 확정한 건 아니고, 다음으로 인력 부족·과잉 비용이 얼마나 생겼는지 계산하자.

In [102]:
holiday_check["staff_gap"] = (
    holiday_check["mean7"] - holiday_check["total"]
) / 2000

holiday_check["shortage_cost"] = (
    -holiday_check["staff_gap"]
).clip(lower=0) * 600_000

holiday_check["surplus_cost"] = (
    holiday_check["staff_gap"]
).clip(lower=0) * 150_000

display(
    holiday_check.groupby("origin")[
        ["shortage_cost", "surplus_cost"]
    ].sum().round(0)
)

,shortage_cost,surplus_cost
origin,,
2025-10-07,165117493.0,15069946.0
2025-10-14,475680.0,42296441.0


In [121]:
comparison_rows = []

for method in ["naive", "seasonal_naive", "mean7"]:
    # 양수는 인력 과잉, 음수는 부족
    gap = (evaluation[method] - evaluation["total"]) / 2000

    shortage_cost = (-gap).clip(lower=0).sum() * 600_000
    surplus_cost = gap.clip(lower=0).sum() * 150_000

    comparison_rows.append({
        "방법": method,
        "MASE": mase_by_pair.loc[
            mase_by_pair["method"].eq(method), "MASE"
        ].mean(),
        "WAPE(%)": (
            evaluation[f"ae_{method}"].sum()
            / evaluation["total"].sum() * 100
        ),
        "적중률(%)": gap.ge(0).mean() * 100,
        "부족 비용(원)": shortage_cost,
        "과잉 비용(원)": surplus_cost,
        "총비용(원)": shortage_cost + surplus_cost
    })

baseline_summary = pd.DataFrame(comparison_rows)
display(baseline_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,naive,1.205,42.840,39.986,1.476971e+09,2.015548e+08,1.678526e+09
1,seasonal_naive,1.245,49.890,50.097,1.326402e+09,3.331332e+08,1.659535e+09
2,mean7,1.218,44.417,58.189,1.191808e+09,2.938619e+08,1.485670e+09


In [122]:
CALIBRATION_WEEKS = 12
corrected_rows = []

for origin in origins:
    for site, data in work_data.groupby("site"):
        series = data.set_index("date")["target_for_model"].sort_index()
        ratios = []

        # 원점 전에 끝난 12개의 주간 예측을 재현
        for k in range(1, CALIBRATION_WEEKS + 1):
            past_origin = origin - pd.Timedelta(days=7 * k)

            past_mean = series.loc[
                (series.index >= past_origin - pd.Timedelta(days=7))
                & (series.index < past_origin)
            ].mean()

            past_actual = series.reindex(
                pd.date_range(past_origin, periods=HORIZON)
            ).dropna()

            if pd.notna(past_mean) and past_mean > 0:
                ratios.extend((past_actual / past_mean).tolist())

        if not ratios:
            raise ValueError(f"보정 자료 없음: {site}, {origin}")

        factor = pd.Series(ratios).quantile(0.8)

        current = baseline_predictions.loc[
            baseline_predictions["origin"].eq(origin)
            & baseline_predictions["site"].eq(site)
        ].copy()

        current["factor"] = factor
        current["mean7_corrected"] = current["mean7"] * factor

        corrected_rows.append(
            current[["origin", "site", "date",
                     "factor", "mean7_corrected"]]
        )

corrected_predictions = pd.concat(corrected_rows, ignore_index=True)

corrected_eval = evaluation.merge(
    corrected_predictions,
    on=["origin", "site", "date"],
    how="left",
    validate="one_to_one"
)

assert corrected_eval["mean7_corrected"].notna().all()

corrected_eval["ae_corrected"] = (
    corrected_eval["mean7_corrected"] - corrected_eval["total"]
).abs()

pair_mae = corrected_eval.groupby(
    ["site", "origin"]
)["ae_corrected"].mean()

pair_scale = scale_table.set_index(["site", "origin"])["scale"]
corrected_mase = (pair_mae / pair_scale).mean()

gap = (
    corrected_eval["mean7_corrected"] - corrected_eval["total"]
) / 2000

shortage = (-gap).clip(lower=0).sum() * 600_000
surplus = gap.clip(lower=0).sum() * 150_000

corrected_summary = pd.DataFrame([{
    "방법": "mean7 + 서비스 수준 보정",
    "MASE": corrected_mase,
    "WAPE(%)": (
        corrected_eval["ae_corrected"].sum()
        / corrected_eval["total"].sum() * 100
    ),
    "적중률(%)": gap.ge(0).mean() * 100,
    "부족 비용(원)": shortage,
    "과잉 비용(원)": surplus,
    "총비용(원)": shortage + surplus
}])

fva_first_two = pd.concat([
    baseline_summary.loc[baseline_summary["방법"].eq("mean7")],
    corrected_summary
], ignore_index=True)

fva_first_two["앞 행 대비 절감액(원)"] = (
    fva_first_two["총비용(원)"].shift()
    - fva_first_two["총비용(원)"]
)

display(fva_first_two.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),앞 행 대비 절감액(원)
0,mean7,1.218,44.417,58.189,1.191808e+09,2.938619e+08,1.485670e+09,NaN
1,mean7 + 서비스 수준 보정,1.967,67.694,79.694,6.130088e+08,7.486928e+08,1.361702e+09,1.239683e+08


In [123]:
def evaluate_predictions(data, accuracy_col, staffing_col, name):
    """현재 임시 평가 규칙: 실제 total > 0인 날만 평가"""
    scored = data.loc[data["total"].gt(0)].copy()

    required = ["total", "scale", accuracy_col, staffing_col]
    if scored[required].isna().any().any():
        raise ValueError("평가 데이터에 결측이 있습니다.")
    if scored["scale"].le(0).any():
        raise ValueError("MASE 분모가 0 이하입니다.")
    if scored[staffing_col].lt(0).any():
        raise ValueError("배치용 예측값이 음수입니다.")

    scored["absolute_error"] = (
        scored[accuracy_col] - scored["total"]
    ).abs()

    # 개소 × 원점별 MAE / 학습 구간 스케일
    pair_scores = scored.groupby(["site", "origin"]).agg(
        MAE=("absolute_error", "mean"),
        scale=("scale", "first")
    )
    pair_scores["MASE"] = pair_scores["MAE"] / pair_scores["scale"]

    gap = (scored[staffing_col] - scored["total"]) / 2000
    shortage = (-gap).clip(lower=0).sum() * 600_000
    surplus = gap.clip(lower=0).sum() * 150_000

    return {
        "방법": name,
        "MASE": pair_scores["MASE"].mean(),
        "WAPE(%)": (
            scored["absolute_error"].sum()
            / scored["total"].sum() * 100
        ),
        "적중률(%)": gap.ge(0).mean() * 100,
        "부족 비용(원)": shortage,
        "과잉 비용(원)": surplus,
        "총비용(원)": shortage + surplus
    }


# 기존 결과를 같은 함수로 다시 계산
common_summary = pd.DataFrame([
    evaluate_predictions(evaluation, "naive", "naive", "naive"),
    evaluate_predictions(
        evaluation, "seasonal_naive", "seasonal_naive",
        "seasonal_naive"
    ),
    evaluate_predictions(evaluation, "mean7", "mean7", "mean7"),
    evaluate_predictions(
        corrected_eval, "mean7_corrected", "mean7_corrected",
        "mean7 + 서비스 수준 보정"
    )
])

# 기존 계산과 일치하는지 확인
import numpy as np

previous = pd.concat(
    [baseline_summary, corrected_summary], ignore_index=True
)
metrics = [
    "MASE", "WAPE(%)", "적중률(%)",
    "부족 비용(원)", "과잉 비용(원)", "총비용(원)"
]

np.testing.assert_allclose(
    common_summary[metrics].to_numpy(dtype=float),
    previous[metrics].to_numpy(dtype=float),
    rtol=1e-10,
    atol=1e-6
)

display(common_summary.round(3))
print("공통 평가 함수: 기존 결과와 일치!")

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,naive,1.205,42.840,39.986,1.476971e+09,2.015548e+08,1.678526e+09
1,seasonal_naive,1.245,49.890,50.097,1.326402e+09,3.331332e+08,1.659535e+09
2,mean7,1.218,44.417,58.189,1.191808e+09,2.938619e+08,1.485670e+09
3,mean7 + 서비스 수준 보정,1.967,67.694,79.694,6.130088e+08,7.486928e+08,1.361702e+09


공통 평가 함수: 기존 결과와 일치!


In [106]:
import torch

print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능: False


In [124]:
tables = []

for method in ["naive", "seasonal_naive", "mean7"]:
    table = baseline_predictions[
        ["origin", "site", "date", method]
    ].rename(columns={method: "point_pred"}).copy()

    table["model"] = method
    table["q80"] = float("nan")  # 단순 예측값은 분위수가 아님
    tables.append(table)

# 서비스 수준 보정: 점예측은 기존 평균, 배치값은 보정값
table = baseline_predictions[
    ["origin", "site", "date", "mean7"]
].rename(columns={"mean7": "point_pred"}).merge(
    corrected_predictions[
        ["origin", "site", "date", "mean7_corrected"]
    ],
    on=["origin", "site", "date"],
    validate="one_to_one"
).rename(columns={"mean7_corrected": "q80"})

table["model"] = "mean7_calibrated"
tables.append(table)

prediction_table = pd.concat(tables, ignore_index=True)[
    ["origin", "site", "date", "model", "point_pred", "q80"]
]

assert not prediction_table.duplicated(
    ["origin", "site", "date", "model"]
).any()

print("결과 행 수:", len(prediction_table))
display(prediction_table.head())

결과 행 수: 33488


,origin,site,date,model,point_pred,q80
0,2025-01-01,경복궁,2025-01-01,naive,17300.0,NaN
1,2025-01-01,경복궁,2025-01-02,naive,17300.0,NaN
2,2025-01-01,경복궁,2025-01-03,naive,17300.0,NaN
3,2025-01-01,경복궁,2025-01-04,naive,17300.0,NaN
4,2025-01-01,경복궁,2025-01-05,naive,17300.0,NaN


In [108]:
from pathlib import Path
from google.colab import files
import shutil

output_dir = Path("outputs/model_validation")
output_dir.mkdir(parents=True, exist_ok=True)

prediction_table.to_csv(
    output_dir / "predictions.csv", index=False, encoding="utf-8-sig"
)
common_summary.to_csv(
    output_dir / "baseline_summary.csv", index=False, encoding="utf-8-sig"
)
mase_by_pair.to_csv(
    output_dir / "baseline_mase_by_site_origin.csv",
    index=False, encoding="utf-8-sig"
)
scale_table.to_csv(
    output_dir / "mase_scales.csv", index=False, encoding="utf-8-sig"
)

archive = shutil.make_archive(
    "model_validation_results", "zip", root_dir=output_dir
)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [110]:
from google.colab import files

uploaded = files.upload()

Saving preprocessed_panel.csv to preprocessed_panel.csv


In [111]:
team_panel = pd.read_csv(
    "preprocessed_panel.csv",
    encoding="utf-8-sig",
    parse_dates=["date"]
)

print("행·열 수:", team_panel.shape)
print("열 목록:", team_panel.columns.tolist())
display(team_panel.head())

행·열 수: (63194, 24)
열 목록: ['date', 'site', 'group', 'weekday', 'closed_weekday', 'is_closed_weekday', 'status', 'is_open', 'y', 'total', 'paid', 'free', 'is_holiday', 'holiday_name', 'seollal_offset', 'chuseok_offset', 'is_last_wednesday', 'is_myeongjeol', 'is_free_rule', 'is_offday', 't2m', 'precip_mm', 'rain', 'in_train_period']


,date,site,group,weekday,closed_weekday,is_closed_weekday,status,is_open,y,total,...,seollal_offset,chuseok_offset,is_last_wednesday,is_myeongjeol,is_free_rule,is_offday,t2m,precip_mm,rain,in_train_period
0,2015-01-01,경복궁,궁,목,화,0,open,1,15644.0,15644.0,...,NaN,NaN,0,0,0,1,-7.77,0.03,0,0
1,2015-01-02,경복궁,궁,금,화,0,open,1,12704.0,12704.0,...,NaN,NaN,0,0,0,0,-6.94,0.31,0,0
2,2015-01-03,경복궁,궁,토,화,0,open,1,13565.0,13565.0,...,NaN,NaN,0,0,0,1,-3.83,0.00,0,0
3,2015-01-04,경복궁,궁,일,화,0,open,1,11218.0,11218.0,...,NaN,NaN,0,0,0,1,1.25,0.03,0,0
4,2015-01-05,경복궁,궁,월,화,0,open,1,7632.0,7632.0,...,NaN,NaN,0,0,0,0,1.77,1.77,0,0


In [112]:
team_recent = team_panel.loc[
    team_panel["date"] >= "2023-01-01"
].copy()

comparison = work_data[
    ["date", "site", "target_for_model"]
].merge(
    team_recent[["date", "site", "y"]],
    on=["date", "site"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

same_value = np.isclose(
    comparison["target_for_model"],
    comparison["y"],
    rtol=0,
    atol=0,
    equal_nan=True
)

different = comparison.loc[
    comparison["_merge"].ne("both") | ~same_value
]

print("팀 데이터의 2023년 이후 행 수:", len(team_recent))
print("날짜·개소 또는 학습용 값 불일치:", len(different))
display(different.head())

팀 데이터의 2023년 이후 행 수: 25208
날짜·개소 또는 학습용 값 불일치: 0


,date,site,target_for_model,y,_merge


In [113]:
model_panel = team_recent.sort_values(
    ["site", "date"]
).reset_index(drop=True).copy()

print("모델용 행 수:", len(model_panel))
print("개소 수:", model_panel["site"].nunique())
print("학습용 y 결측:", model_panel["y"].isna().sum())

모델용 행 수: 25208
개소 수: 23
학습용 y 결측: 3700


In [114]:
HORIZON = 7
origins = pd.date_range(
    "2025-01-01", periods=52, freq="7D"
)

print("원점 수:", len(origins))
print("마지막 예측일:",
      origins[-1] + pd.Timedelta(days=6))

원점 수: 52
마지막 예측일: 2025-12-30 00:00:00


In [125]:
archive = shutil.make_archive(
    "model_validation_2025_52origins",
    "zip",
    root_dir=output_dir
)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import torch

print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능: True
GPU 이름: Tesla T4


In [2]:
%pip install -U chronos-forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 4.6 MB/s eta 0:00:00


In [3]:
from chronos import Chronos2Pipeline
from importlib.metadata import version

print("Chronos 버전:", version("chronos-forecasting"))
print("Chronos-2 불러오기 성공!")

Chronos 버전: 2.3.2
Chronos-2 불러오기 성공!


In [4]:
import torch
from chronos import Chronos2Pipeline

assert torch.cuda.is_available(), "GPU 연결을 확인해주세요."

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda"
)

print("Chronos-2 모델 준비 완료!")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos-2 모델 준비 완료!


In [5]:
required = [
    "model_panel", "origins", "scale_table",
    "evaluation", "evaluate_predictions"
]

for name in required:
    print(name, "준비됨" if name in globals() else "없음")

model_panel 없음
origins 없음
scale_table 없음
evaluation 없음
evaluate_predictions 없음


In [6]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

Saving preprocessed_panel.csv to preprocessed_panel.csv


In [7]:
team_panel = pd.read_csv(
    "preprocessed_panel.csv",
    encoding="utf-8-sig",
    parse_dates=["date"]
)

model_panel = (
    team_panel.loc[team_panel["date"] >= "2023-01-01"]
    .sort_values(["site", "date"])
    .reset_index(drop=True)
    .copy()
)

HORIZON = 7
origins = pd.date_range("2025-01-01", periods=52, freq="7D")

assert not model_panel.duplicated(["site", "date"]).any()

print("모델용 행 수:", len(model_panel))
print("개소 수:", model_panel["site"].nunique())
print("y 결측:", model_panel["y"].isna().sum())
print("원점 수:", len(origins))

모델용 행 수: 25208
개소 수: 23
y 결측: 3700
원점 수: 52


In [8]:
test_origin = origins[0]
site_order = sorted(model_panel["site"].unique())
contexts = []

for site in site_order:
    history = model_panel.loc[
        model_panel["site"].eq(site)
        & model_panel["date"].lt(test_origin),
        ["date", "y"]
    ].set_index("date")["y"]

    # 일별 날짜 간격 유지
    history = history.reindex(
        pd.date_range("2023-01-01",
                      test_origin - pd.Timedelta(days=1))
    )

    contexts.append(
        torch.tensor(history.to_numpy(dtype="float32"))
    )

with torch.inference_mode():
    quantiles, _ = pipeline.predict_quantiles(
        inputs=contexts,
        prediction_length=HORIZON,
        quantile_levels=[0.5, 0.8]
    )

test_rows = []

for site, forecast in zip(site_order, quantiles):
    values = forecast.detach().cpu().numpy().reshape(HORIZON, 2)

    for date, (q50, q80) in zip(
        pd.date_range(test_origin, periods=HORIZON), values
    ):
        test_rows.append({
            "origin": test_origin,
            "site": site,
            "date": date,
            "model": "chronos2_zero_shot",
            "point_pred": max(float(q50), 0),
            "q80": max(float(q80), 0)
        })

chronos_test = pd.DataFrame(test_rows)

assert np.isfinite(
    chronos_test[["point_pred", "q80"]].to_numpy()
).all()

print("예측 행 수:", len(chronos_test))
display(chronos_test.head())

예측 행 수: 161


,origin,site,date,model,point_pred,q80
0,2025-01-01,경복궁,2025-01-01,chronos2_zero_shot,15298.055664,19315.173828
1,2025-01-01,경복궁,2025-01-02,chronos2_zero_shot,11637.186523,14578.318359
2,2025-01-01,경복궁,2025-01-03,chronos2_zero_shot,12024.636719,14985.455078
3,2025-01-01,경복궁,2025-01-04,chronos2_zero_shot,15590.753906,19700.669922
4,2025-01-01,경복궁,2025-01-05,chronos2_zero_shot,13939.369141,17448.857422


In [9]:
from pathlib import Path
from tqdm.auto import tqdm

checkpoint_dir = Path("outputs/chronos2_zero_shot")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# 첫 원점 결과 보관
chronos_test.to_csv(
    checkpoint_dir / f"{origins[0]:%Y-%m-%d}.csv",
    index=False
)

series_by_site = {
    site: model_panel.loc[model_panel["site"].eq(site)]
    .set_index("date")["y"].sort_index()
    for site in site_order
}

all_forecasts = []

for origin in tqdm(origins, desc="Chronos 원점별 예측"):
    checkpoint = checkpoint_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        dates = pd.date_range(
            "2023-01-01", origin - pd.Timedelta(days=1)
        )
        contexts = [
            torch.tensor(
                series_by_site[site].reindex(dates)
                .to_numpy(dtype="float32")
            )
            for site in site_order
        ]

        with torch.inference_mode():
            quantiles, _ = pipeline.predict_quantiles(
                inputs=contexts,
                prediction_length=HORIZON,
                quantile_levels=[0.5, 0.8]
            )

        rows = []
        for site, forecast in zip(site_order, quantiles):
            values = (
                forecast.detach().cpu().numpy()
                .reshape(HORIZON, 2)
            )
            for date, (q50, q80) in zip(
                pd.date_range(origin, periods=HORIZON), values
            ):
                rows.append({
                    "origin": origin,
                    "site": site,
                    "date": date,
                    "model": "chronos2_zero_shot",
                    "point_pred": max(float(q50), 0),
                    "q80": max(float(q80), 0)
                })

        result = pd.DataFrame(rows)

        assert np.isfinite(
            result[["point_pred", "q80"]].to_numpy()
        ).all()
        assert result["q80"].ge(result["point_pred"]).all()

        result.to_csv(checkpoint, index=False)

    all_forecasts.append(result)

chronos_predictions = pd.concat(all_forecasts, ignore_index=True)

assert len(chronos_predictions) == 52 * 23 * 7
assert not chronos_predictions.duplicated(
    ["origin", "site", "date"]
).any()

print("전체 예측 행 수:", len(chronos_predictions))
print("원점 수:", chronos_predictions["origin"].nunique())

Chronos 원점별 예측:   0%|          | 0/52 [00:00<?, ?it/s]

전체 예측 행 수: 8372
원점 수: 52


## Chronos-2 제로샷 실행 기록 · 2026-10-14

### 실행 목적
과거 관람객만으로 예측하는 사전학습 모델을 베이스라인과 같은 조건에서 비교한다. **추가 학습·달력·날씨 공변량은 사용하지 않는다.**

### 실행 설정

| 항목 | 설정 |
|---|---|
| 패키지 | `chronos-forecasting 2.3.2` |
| 모델 | `amazon/chronos-2` |
| 실행 환경 | Colab · Tesla T4 |
| 대상 | 궁·능 23개소 |
| 과거 입력 범위 | 2023-01-01부터 각 원점 전날까지 전달 |
| 백테스트 원점 | 2025-01-01부터 7일 간격, 52개 |
| 예측 기간 | 원점마다 7일 |
| 평가 예정 날짜 | 2025-01-01~12-30 — **12/31 제외** |
| 점예측 | 0.5 분위수 |
| 인력 배치용 예측 | 0.8 분위수 |

※ 실제 모델이 사용하는 입력 길이는 모델 기본 context 설정에 따라 제한될 수 있다. 사용 설정은 이후 재현 기록에 남긴다.

### 데이터 처리
- 팀 전처리 파일의 `y`를 사용한다.
- 날짜 간격을 유지하고, 휴관·결측의 NaN을 삭제하거나 0으로 채우지 않는다.
- 음수 예측은 0으로 제한한다.
- 미래의 실제 관람객·날씨·개방 상태는 입력하지 않는다.

### 현재 진행
- GPU 연결, 패키지 설치, 모델 로딩 **완료**
- 팀 데이터 복구: **25,208행·23개소·y 결측 3,700개**
- 첫 원점 **2025-01-01** 시험 예측 **성공: 161행**
- 전체 52개 원점 실행 코드를 준비했다. **완료 여부는 실행 결과로 확인한다.**
- 전체 실행 완료 시 예상 결과: **8,372행**

### 저장·평가
- 원점별 예측을 `outputs/chronos2_zero_shot/`에 저장한다.
- 저장 파일은 런타임 임시 파일이므로 완료 후 별도 다운로드한다.
- **점예측으로 MASE·WAPE**, **0.8 분위수로 적중률·배치 비용**을 평가한다.
- 현재는 실행 확인 단계이며, **Chronos 성능·비용 개선 여부는 아직 평가 전**이다.

In [10]:
chronos_predictions.to_csv(
    "chronos2_zero_shot_2025_52origins.csv",
    index=False,
    encoding="utf-8-sig"
)
files.download("chronos2_zero_shot_2025_52origins.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Chronos 예측 결과 저장

**파일:** `chronos2_zero_shot_2025_52origins.csv`

- **내용:** Chronos-2가 원점 52개에서 각각 23개소의 다음 7일 관람객을 예측한 결과
- **규모:** 52개 원점 × 23개소 × 7일 = **8,372행**
- **예측 날짜:** 2025-01-01~12-30

| 열 | 의미 |
|---|---|
| `origin` | 예측 기간의 첫날. 전날까지의 관람객 기록을 입력 |
| `site` | 예측한 개소 |
| `date` | 관람객을 예측한 날짜 |
| `model` | 사용한 모델 이름 |
| `point_pred` | 0.5 분위수 예측값 → MASE·WAPE 평가에 사용 |
| `q80` | 0.8 분위수 예측값 → 인력 배치·적중률·비용 계산에 사용 |

**용도:** 실제 관람객과 비교해 모델을 평가하고, 날짜·개소별 오차와 인력 부족을 분석한다. 예측을 다시 실행하지 않고 결과를 재사용하고 팀원과 공유할 수 있다.

※ 이 파일은 **과거 백테스트용 예측 결과**이며, 최종 제출할 2026년 인력 배치 권고표는 아니다.

In [11]:
lag7 = model_panel[["date", "site", "y"]].copy()
lag7["date"] += pd.Timedelta(days=7)
lag7 = lag7.rename(columns={"y": "y_lag7"})

scale_data = model_panel.merge(
    lag7, on=["date", "site"],
    how="left", validate="one_to_one"
)
scale_data["weekly_diff"] = (
    scale_data["y"] - scale_data["y_lag7"]
).abs()

scale_table = pd.concat([
    scale_data.loc[scale_data["date"] < origin]
    .groupby("site")["weekly_diff"].mean()
    .reindex(site_order)
    .rename("scale")
    .reset_index()
    .assign(origin=origin)
    for origin in origins
], ignore_index=True)

chronos_eval = chronos_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left", validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left", validate="many_to_one"
)

assert scale_table["scale"].notna().all()
assert scale_table["scale"].gt(0).all()
assert chronos_eval["total"].notna().all()

print("MASE分母:", len(scale_table))
print("評価準備行数:", len(chronos_eval))

MASE分母: 1196
評価準備行数: 8372


In [12]:
chronos_summary = pd.DataFrame([
    evaluate_predictions(
        chronos_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos-2 제로샷"
    )
])

display(chronos_summary.round(3))

NameError: name 'evaluate_predictions' is not defined

In [13]:
def evaluate_predictions(data, accuracy_col, staffing_col, name):
    scored = data.loc[data["total"].gt(0)].copy()

    required = ["total", "scale", accuracy_col, staffing_col]
    if scored[required].isna().any().any():
        raise ValueError("평가 데이터에 결측이 있습니다.")
    if scored["scale"].le(0).any():
        raise ValueError("MASE 분모가 0 이하입니다.")
    if scored[staffing_col].lt(0).any():
        raise ValueError("배치용 예측값이 음수입니다.")

    scored["absolute_error"] = (
        scored[accuracy_col] - scored["total"]
    ).abs()

    pair_scores = scored.groupby(["site", "origin"]).agg(
        MAE=("absolute_error", "mean"),
        scale=("scale", "first")
    )
    pair_scores["MASE"] = (
        pair_scores["MAE"] / pair_scores["scale"]
    )

    gap = (scored[staffing_col] - scored["total"]) / 2000
    shortage = (-gap).clip(lower=0).sum() * 600_000
    surplus = gap.clip(lower=0).sum() * 150_000

    return {
        "방법": name,
        "MASE": pair_scores["MASE"].mean(),
        "WAPE(%)": (
            scored["absolute_error"].sum()
            / scored["total"].sum() * 100
        ),
        "적중률(%)": gap.ge(0).mean() * 100,
        "부족 비용(원)": shortage,
        "과잉 비용(원)": surplus,
        "총비용(원)": shortage + surplus
    }


chronos_summary = pd.DataFrame([
    evaluate_predictions(
        chronos_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos-2 제로샷"
    )
])

display(chronos_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,Chronos-2 제로샷,0.85,31.836,80.501,6.491375e+08,3.550995e+08,1.004237e+09


# 「Chronos-2 제로샷 vs 베이스라인 · 1차 비교」
- 평가: 2025년 원점 52개·7일 예측, 실제 관람객이 있는 날
- 정확도: 점예측 기준 / 비용·적중률: 배치용 값 기준
- Chronos 입력: 과거 관람객만 사용, 달력·날씨 미포함
- 현재 결론: 전체 평균에서 개선 확인. 무료일·명절별 성능과 최종 모델 선택은 추가 검증 예정
기존 보정값 MASE 1.967과 이번 표의 1.218은 평가한 값이 다르다

In [14]:
chronos_by_type = chronos_eval.merge(
    model_panel[
        ["date", "site", "is_myeongjeol",
         "is_free_rule", "is_holiday"]
    ],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
)

# 겹치는 날은 명절 → 무료일 → 공휴일 순으로 분류
chronos_by_type["date_type"] = np.select(
    [
        chronos_by_type["is_myeongjeol"].eq(1),
        chronos_by_type["is_free_rule"].eq(1),
        chronos_by_type["is_holiday"].eq(1)
    ],
    [
        "명절 연휴",
        "무료 개방일(명절 제외)",
        "공휴일(명절·무료일 제외)"
    ],
    default="그 외 개방일"
)

type_results = []

for date_type, data in chronos_by_type.groupby("date_type"):
    result = evaluate_predictions(
        data,
        accuracy_col="point_pred",
        staffing_col="q80",
        name=date_type
    )
    result["평가건수"] = data["total"].gt(0).sum()
    type_results.append(result)

chronos_type_summary = pd.DataFrame(type_results)
display(chronos_type_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),평가건수
0,공휴일(명절·무료일 제외),1.944,40.062,32.367,4.009155e+07,4.245995e+06,4.433755e+07,207
1,그 외 개방일,0.684,26.508,85.679,2.441160e+08,3.436348e+08,5.877508e+08,6445
2,명절 연휴,4.356,61.997,31.104,2.864172e+08,3.885011e+06,2.903022e+08,299
3,무료 개방일(명절 제외),1.305,44.805,42.795,7.851273e+07,3.333736e+06,8.184646e+07,229


# 「전체 평균에 가려진 명절·무료일 인력 부족」
Chronos-2 제로샷의 전체 적중률은 80.5%였지만, 명절 연휴에서는 31.1%였다. 명절은 전체 평가건수의 약 4.2%인데 인력 부족 비용의 약 44.1%를 차지했다. 따라서 전체 성능뿐 아니라 날짜 유형별 성능을 확인하고, 명절·무료 개방 정보를 반영한 예측을 추가 검증할 필요가 있다.

‘그 외 개방일’에는 주말도 포함, 무료일은 팀 달력 규칙 기준

In [15]:
# 모델 입력에는 실제값으로 만든 status·is_open을 넣지 않음
calendar_panel = model_panel[["date", "site", "y"]].copy()

# 요일은 숫자의 크기가 아니라 범주로 전달
for day in range(7):
    calendar_panel[f"dow_{day}"] = (
        calendar_panel["date"].dt.dayofweek == day
    ).astype(float)

calendar_panel["regular_closed_day"] = (
    model_panel["weekday"] == model_panel["closed_weekday"]
).astype(float).to_numpy()

feature_cols = [
    *[f"dow_{day}" for day in range(7)],
    "regular_closed_day"
]

test_origin = origins[0]

context_df = calendar_panel.loc[
    calendar_panel["date"] < test_origin
].copy()

future_df = calendar_panel.loc[
    calendar_panel["date"].between(
        test_origin,
        test_origin + pd.Timedelta(days=HORIZON - 1)
    ),
    ["site", "date", *feature_cols]
].copy()  # 미래 y는 전달하지 않음

calendar_test = pipeline.predict_df(
    df=context_df,
    future_df=future_df,
    id_column="site",
    timestamp_column="date",
    target="y",
    prediction_length=HORIZON,
    quantile_levels=[0.5, 0.8],
    cross_learning=False,
    freq="D"
)

print("예측 행 수:", len(calendar_test))
print("출력 열:", calendar_test.columns.tolist())
display(calendar_test.head())

예측 행 수: 161
출력 열: ['site', 'date', 'target_name', 'predictions', '0.5', '0.8']


,site,date,target_name,predictions,0.5,0.8
0,경복궁,2025-01-01,y,16485.142578,16485.142578,20352.486328
1,경복궁,2025-01-02,y,12920.447266,12920.447266,15804.691406
2,경복궁,2025-01-03,y,13029.853516,13029.853516,15883.677734
3,경복궁,2025-01-04,y,16425.324219,16425.324219,21111.064453
4,경복궁,2025-01-05,y,14863.002930,14863.002930,18832.464844


In [16]:
calendar_dir = Path("outputs/chronos2_calendar")
calendar_dir.mkdir(parents=True, exist_ok=True)

def format_calendar_result(raw, origin):
    result = raw.rename(columns={
        "0.5": "point_pred",
        "0.8": "q80"
    })[["site", "date", "point_pred", "q80"]].copy()

    result["date"] = pd.to_datetime(result["date"])
    result["origin"] = origin
    result["model"] = "chronos2_weekday_regular_closure"

    result[["point_pred", "q80"]] = (
        result[["point_pred", "q80"]].clip(lower=0)
    )

    assert len(result) == 23 * HORIZON
    assert not result.duplicated(["site", "date"]).any()
    assert np.isfinite(
        result[["point_pred", "q80"]].to_numpy()
    ).all()
    assert result["q80"].ge(result["point_pred"]).all()

    return result[
        ["origin", "site", "date", "model", "point_pred", "q80"]
    ]

# 시험한 첫 원점은 재사용
format_calendar_result(calendar_test, origins[0]).to_csv(
    calendar_dir / f"{origins[0]:%Y-%m-%d}.csv",
    index=False
)

calendar_results = []

for origin in tqdm(origins, desc="요일·휴관 정보 추가"):
    checkpoint = calendar_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        context_df = calendar_panel.loc[
            calendar_panel["date"] < origin
        ].copy()

        future_df = calendar_panel.loc[
            calendar_panel["date"].between(
                origin,
                origin + pd.Timedelta(days=HORIZON - 1)
            ),
            ["site", "date", *feature_cols]
        ].copy()

        raw = pipeline.predict_df(
            df=context_df,
            future_df=future_df,
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=HORIZON,
            quantile_levels=[0.5, 0.8],
            cross_learning=False,
            freq="D"
        )

        result = format_calendar_result(raw, origin)
        result.to_csv(checkpoint, index=False)

    calendar_results.append(result)

chronos_calendar_predictions = pd.concat(
    calendar_results, ignore_index=True
)

assert len(chronos_calendar_predictions) == 8372

print("전체 예측 행 수:", len(chronos_calendar_predictions))
print("원점 수:",
      chronos_calendar_predictions["origin"].nunique())

요일·휴관 정보 추가:   0%|          | 0/52 [00:00<?, ?it/s]

전체 예측 행 수: 8372
원점 수: 52


# 과거 관람객 + 요일 + 정기 휴관 정보, 공휴일·명절은 미포함

In [18]:
calendar_eval = chronos_calendar_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left",
    validate="many_to_one"
)

calendar_summary = pd.DataFrame([
    evaluate_predictions(
        calendar_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos + 요일·정기 휴관"
    )
])

display(pd.concat(
    [chronos_summary, calendar_summary],
    ignore_index=True
).round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,Chronos-2 제로샷,0.850,31.836,80.501,6.491375e+08,3.550995e+08,1.004237e+09
1,Chronos + 요일·정기 휴관,0.872,32.612,80.738,6.316125e+08,3.864082e+08,1.018021e+09


요일·정기 휴관 정보를 추가했을 때 MASE와 총비용이 소폭 증가했다. 이 설정에서는 정보 추가의 개선 효과가 확인되지 않았다.

In [19]:
feature_check = model_panel.loc[
    model_panel["date"] < "2025-01-01"
].copy()

feature_check["year"] = feature_check["date"].dt.year
feature_check["month"] = feature_check["date"].dt.month

# 비교 기준: 공휴일·명절·무료일이 아닌 개방일
normal = feature_check.loc[
    feature_check["y"].notna()
    & feature_check["is_holiday"].eq(0)
    & feature_check["is_myeongjeol"].eq(0)
    & feature_check["is_free_rule"].eq(0)
]

keys = ["site", "year", "month", "weekday"]

normal_means = (
    normal.groupby(keys)["y"].mean()
    .rename("normal_mean").reset_index()
)

offset_results = []

for holiday, offset_col in [
    ("설", "seollal_offset"),
    ("추석", "chuseok_offset")
]:
    selected = feature_check.loc[
        feature_check["y"].notna()
        & feature_check[offset_col].isin([-1, 0, 1])
    ].merge(
        normal_means,
        on=keys,
        how="left",
        validate="many_to_one"
    )

    selected = selected.loc[selected["normal_mean"].gt(0)].copy()
    selected["ratio"] = selected["y"] / selected["normal_mean"]

    summary = (
        selected.groupby(offset_col)["ratio"]
        .agg(비교건수="count", 배수_중앙값="median")
        .reset_index()
        .rename(columns={offset_col: "offset"})
    )
    summary["명절"] = holiday
    offset_results.append(summary)

offset_effects = pd.concat(offset_results, ignore_index=True)
offset_effects["위치"] = offset_effects["offset"].map(
    {-1: "전날", 0: "당일", 1: "다음날"}
)

display(offset_effects[
    ["명절", "위치", "비교건수", "배수_중앙값"]
].round(3))

,명절,위치,비교건수,배수_중앙값
0,설,전날,46,1.723
1,설,당일,46,1.778
2,설,다음날,25,1.974
3,추석,전날,25,3.124
4,추석,당일,44,4.234
5,추석,다음날,46,3.009


In [20]:
holiday_panel = calendar_panel.merge(
    model_panel[
        ["date", "site", "is_holiday", "is_myeongjeol"]
    ],
    on=["date", "site"],
    how="left",
    validate="one_to_one"
)

holiday_features = [
    *feature_cols,
    "is_holiday",
    "is_myeongjeol"
]

test_origin = origins[0]

holiday_test = pipeline.predict_df(
    df=holiday_panel.loc[
        holiday_panel["date"] < test_origin
    ].copy(),
    future_df=holiday_panel.loc[
        holiday_panel["date"].between(
            test_origin,
            test_origin + pd.Timedelta(days=HORIZON - 1)
        ),
        ["site", "date", *holiday_features]
    ].copy(),
    id_column="site",
    timestamp_column="date",
    target="y",
    prediction_length=HORIZON,
    quantile_levels=[0.5, 0.8],
    cross_learning=False,
    freq="D"
)

print("예측 행 수:", len(holiday_test))
display(holiday_test.head())

예측 행 수: 161


,site,date,target_name,predictions,0.5,0.8
0,경복궁,2025-01-01,y,17267.623047,17267.623047,21641.707031
1,경복궁,2025-01-02,y,13050.398438,13050.398438,15936.314453
2,경복궁,2025-01-03,y,13235.982422,13235.982422,16132.352539
3,경복궁,2025-01-04,y,16103.270508,16103.270508,20418.531250
4,경복궁,2025-01-05,y,14650.578125,14650.578125,18213.787109


In [21]:
holiday_dir = Path("outputs/chronos2_holiday")
holiday_dir.mkdir(parents=True, exist_ok=True)

def format_holiday_result(raw, origin):
    result = format_calendar_result(raw, origin)
    result["model"] = "chronos2_weekday_holiday"
    return result

format_holiday_result(holiday_test, origins[0]).to_csv(
    holiday_dir / f"{origins[0]:%Y-%m-%d}.csv",
    index=False
)

holiday_results = []

for origin in tqdm(origins, desc="공휴일·명절 정보 추가"):
    checkpoint = holiday_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        raw = pipeline.predict_df(
            df=holiday_panel.loc[
                holiday_panel["date"] < origin
            ].copy(),
            future_df=holiday_panel.loc[
                holiday_panel["date"].between(
                    origin,
                    origin + pd.Timedelta(days=HORIZON - 1)
                ),
                ["site", "date", *holiday_features]
            ].copy(),
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=HORIZON,
            quantile_levels=[0.5, 0.8],
            cross_learning=False,
            freq="D"
        )

        result = format_holiday_result(raw, origin)
        result.to_csv(checkpoint, index=False)

    holiday_results.append(result)

chronos_holiday_predictions = pd.concat(
    holiday_results, ignore_index=True
)

assert len(chronos_holiday_predictions) == 8372
assert not chronos_holiday_predictions.duplicated(
    ["origin", "site", "date"]
).any()

holiday_eval = chronos_holiday_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left", validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left", validate="many_to_one"
)

holiday_summary = pd.DataFrame([
    evaluate_predictions(
        holiday_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos + 요일・휴관 + 공휴일・명절"
    )
])

display(pd.concat(
    [chronos_summary, calendar_summary, holiday_summary],
    ignore_index=True
).round(3))

공휴일·명절 정보 추가:   0%|          | 0/52 [00:00<?, ?it/s]

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,Chronos-2 제로샷,0.850,31.836,80.501,6.491375e+08,3.550995e+08,1.004237e+09
1,Chronos + 요일·정기 휴관,0.872,32.612,80.738,6.316125e+08,3.864082e+08,1.018021e+09
2,Chronos + 요일・휴관 + 공휴일・명절,0.840,30.287,80.836,4.749955e+08,4.035375e+08,8.785330e+08


In [22]:
offset_panel = holiday_panel.merge(
    model_panel[
        ["date", "site", "seollal_offset", "chuseok_offset"]
    ],
    on=["date", "site"],
    how="left",
    validate="one_to_one"
)

offset_features = holiday_features.copy()

for col in ["seollal_offset", "chuseok_offset"]:
    near_col = f"{col}_is_near"

    # 0이 '당일'인지 '명절 근처가 아님'인지 구분
    offset_panel[near_col] = offset_panel[col].notna().astype(float)

    # -15~15를 -1~1로 고정 변환: 학습 통계를 쓰지 않음
    offset_panel[col] = offset_panel[col].fillna(0) / 15.0

    offset_features.extend([col, near_col])

assert offset_panel[offset_features].notna().all().all()

print("offset 추가 완료")
print("추가 변수:",
      [x for x in offset_features if x not in holiday_features])

offset 추가 완료
추가 변수: ['seollal_offset', 'seollal_offset_is_near', 'chuseok_offset', 'chuseok_offset_is_near']


In [23]:
offset_dir = Path("outputs/chronos2_offset")
offset_dir.mkdir(parents=True, exist_ok=True)

offset_results = []

for origin in tqdm(origins, desc="설·추석 offset 추가"):
    checkpoint = offset_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        raw = pipeline.predict_df(
            df=offset_panel.loc[
                offset_panel["date"] < origin
            ].copy(),
            future_df=offset_panel.loc[
                offset_panel["date"].between(
                    origin,
                    origin + pd.Timedelta(days=HORIZON - 1)
                ),
                ["site", "date", *offset_features]
            ].copy(),
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=HORIZON,
            quantile_levels=[0.5, 0.8],
            cross_learning=False,
            freq="D"
        )

        result = format_calendar_result(raw, origin)
        result["model"] = "chronos2_holiday_offset"
        result.to_csv(checkpoint, index=False)

    offset_results.append(result)

chronos_offset_predictions = pd.concat(
    offset_results, ignore_index=True
)

assert len(chronos_offset_predictions) == 8372
assert not chronos_offset_predictions.duplicated(
    ["origin", "site", "date"]
).any()

offset_eval = chronos_offset_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left", validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left", validate="many_to_one"
)

offset_summary = pd.DataFrame([
    evaluate_predictions(
        offset_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos + 曜日・휴관 + 공휴일・명절 + offset"
    )
])

display(pd.concat(
    [chronos_summary, calendar_summary,
     holiday_summary, offset_summary],
    ignore_index=True
).round(3))

설·추석 offset 추가:   0%|          | 0/52 [00:00<?, ?it/s]

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,Chronos-2 제로샷,0.850,31.836,80.501,6.491375e+08,3.550995e+08,1.004237e+09
1,Chronos + 요일·정기 휴관,0.872,32.612,80.738,6.316125e+08,3.864082e+08,1.018021e+09
2,Chronos + 요일・휴관 + 공휴일・명절,0.840,30.287,80.836,4.749955e+08,4.035375e+08,8.785330e+08
3,Chronos + 曜日・휴관 + 공휴일・명절 + offset,0.837,30.056,80.850,4.889678e+08,3.914012e+08,8.803690e+08


In [24]:
free_panel = offset_panel.merge(
    model_panel[["date", "site", "is_free_rule"]],
    on=["date", "site"],
    how="left",
    validate="one_to_one"
)

free_features = [*offset_features, "is_free_rule"]
assert free_panel[free_features].notna().all().all()

free_dir = Path("outputs/chronos2_free_rule")
free_dir.mkdir(parents=True, exist_ok=True)

free_results = []

for origin in tqdm(origins, desc="무료 개방일 정보 추가"):
    checkpoint = free_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        raw = pipeline.predict_df(
            df=free_panel.loc[
                free_panel["date"] < origin
            ].copy(),
            future_df=free_panel.loc[
                free_panel["date"].between(
                    origin,
                    origin + pd.Timedelta(days=HORIZON - 1)
                ),
                ["site", "date", *free_features]
            ].copy(),
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=HORIZON,
            quantile_levels=[0.5, 0.8],
            cross_learning=False,
            freq="D"
        )

        result = format_calendar_result(raw, origin)
        result["model"] = "chronos2_holiday_offset_free"
        result.to_csv(checkpoint, index=False)

    free_results.append(result)

chronos_free_predictions = pd.concat(
    free_results, ignore_index=True
)

assert len(chronos_free_predictions) == 8372
assert not chronos_free_predictions.duplicated(
    ["origin", "site", "date"]
).any()

free_eval = chronos_free_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left", validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left", validate="many_to_one"
)

free_summary = pd.DataFrame([
    evaluate_predictions(
        free_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos + 요일·휴관 + 공휴일·명절 + offset + 무료일"
    )
])

chronos_fva_summary = pd.concat(
    [chronos_summary, calendar_summary,
     holiday_summary, offset_summary, free_summary],
    ignore_index=True
)

chronos_fva_summary["앞 행 대비 절감액(원)"] = (
    chronos_fva_summary["총비용(원)"].shift()
    - chronos_fva_summary["총비용(원)"]
)

display(chronos_fva_summary.round(3))

무료 개방일 정보 추가:   0%|          | 0/52 [00:00<?, ?it/s]

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),앞 행 대비 절감액(원)
0,Chronos-2 제로샷,0.850,31.836,80.501,6.491375e+08,3.550995e+08,1.004237e+09,NaN
1,Chronos + 요일·정기 휴관,0.872,32.612,80.738,6.316125e+08,3.864082e+08,1.018021e+09,-1.378372e+07
2,Chronos + 요일・휴관 + 공휴일・명절,0.840,30.287,80.836,4.749955e+08,4.035375e+08,8.785330e+08,1.394877e+08
3,Chronos + 曜日・휴관 + 공휴일・명절 + offset,0.837,30.056,80.850,4.889678e+08,3.914012e+08,8.803690e+08,-1.835932e+06
4,Chronos + 요일·휴관 + 공휴일·명절 + offset + 무료일,0.822,29.148,81.170,4.383769e+08,3.901462e+08,8.285232e+08,5.184581e+07


## Chronos · 달력 정보 추가 효과

**공통 조건:** 2023년부터 원점 전날까지의 기록을 입력하고, 2025년 원점 52개에서 각각 7일을 예측했다. 평가는 1월 1일~12월 30일 중 실제 관람객이 있는 날을 대상으로 했다.

모든 단계는 **추가 학습 없는 제로샷**이다. MASE·WAPE는 0.5 분위수, 적중률·비용은 0.8 분위수로 계산했다.

| 추가한 정보 | MASE ↓ | WAPE ↓ | 적중률 | 총비용 | 앞 단계 대비 비용 변화 |
|---|---:|---:|---:|---:|---|
| 과거 관람객만 | 0.850 | 31.836% | 80.501% | 약 10.04억 원 | — |
| + 요일·정기 휴관 | 0.872 | 32.612% | 80.738% | 약 10.18억 원 | 약 1,378만 원 증가 |
| + 공휴일·명절 | 0.840 | 30.287% | 80.836% | 약 8.79억 원 | 약 1억 3,949만 원 절감 |
| + 설·추석 offset 및 근처 표시 | 0.837 | 30.056% | 80.850% | 약 8.80억 원 | 약 184만 원 증가 |
| + 무료 개방일 규칙 | **0.822** | **29.148%** | **81.170%** | **약 8.29억 원** | **약 5,185만 원 절감** |

### 확인한 내용

- 정보 추가가 항상 개선으로 이어지지는 않았다. 요일·휴관 단계와 offset 단계의 비용 증가도 기록했다.
- 공휴일·명절 정보 추가 후 부족 비용이 크게 감소했다.
- 무료일 정보 추가 후 MASE와 비용이 모두 개선됐다.
- 과거 관람객만 사용하는 Chronos 대비 **총비용 약 1억 7,571만 원, 17.5% 절감**을 확인했다.

### 해석 범위

무료일 변수는 팀의 `is_free_rule`, 즉 **명절 연휴 또는 매월 마지막 수요일**이다. 앞 단계에 명절 여부가 이미 있으므로, 이번 단계에서 새로 구분되는 날짜는 주로 마지막 수요일이다. 임시 무료 개방 공지는 아직 반영하지 않았다.

현재 결과는 전체 평균이다. **명절·무료일별 성능 개선과 다른 후보 모델 비교를 확인한 뒤 최종 모델을 선택한다.**

In [25]:
free_by_type = free_eval.merge(
    chronos_by_type[["origin", "site", "date", "date_type"]],
    on=["origin", "site", "date"],
    how="left",
    validate="one_to_one"
)

assert free_by_type["date_type"].notna().all()

free_type_results = []

for date_type, data in free_by_type.groupby("date_type"):
    result = evaluate_predictions(
        data,
        accuracy_col="point_pred",
        staffing_col="q80",
        name=date_type
    )
    result["평가건수"] = data["total"].gt(0).sum()
    free_type_results.append(result)

free_type_summary = pd.DataFrame(free_type_results)

type_comparison = chronos_type_summary.merge(
    free_type_summary,
    on="방법",
    suffixes=("_기본", "_달력추가"),
    validate="one_to_one"
)

assert (
    type_comparison["평가건수_기본"]
    == type_comparison["평가건수_달력추가"]
).all()

type_comparison["비용 절감액(원)"] = (
    type_comparison["총비용(원)_기본"]
    - type_comparison["총비용(원)_달력추가"]
)

display(type_comparison[[
    "방법", "평가건수_기본",
    "MASE_기본", "MASE_달력추가",
    "적중률(%)_기본", "적중률(%)_달력추가",
    "비용 절감액(원)"
]].round(3))

,방법,평가건수_기본,MASE_기본,MASE_달력추가,적중률(%)_기본,적중률(%)_달력추가,비용 절감액(원)
0,공휴일(명절·무료일 제외),207,1.944,1.828,32.367,37.198,5.804662e+06
1,그 외 개방일,6445,0.684,0.685,85.679,84.003,1.287990e+07
2,명절 연휴,299,4.356,3.654,31.104,66.890,1.209598e+08
3,무료 개방일(명절 제외),229,1.305,1.028,42.795,59.825,3.606952e+07


In [26]:
shortage_cases = free_eval.loc[
    free_eval["total"].gt(0)
].copy()

shortage_cases["필요 인력"] = shortage_cases["total"] / 2000
shortage_cases["배치 인력"] = shortage_cases["q80"] / 2000
shortage_cases["부족 인원"] = (
    shortage_cases["필요 인력"]
    - shortage_cases["배치 인력"]
).clip(lower=0)

shortage_cases["부족 비용(원)"] = (
    shortage_cases["부족 인원"] * 600_000
)

top10_shortages = (
    shortage_cases.nlargest(10, "부족 인원")
    .merge(
        model_panel[[
            "date", "site", "is_holiday",
            "is_myeongjeol", "is_free_rule"
        ]],
        on=["date", "site"],
        how="left",
        validate="many_to_one"
    )
)

display(top10_shortages[[
    "origin", "date", "site", "total",
    "point_pred", "q80",
    "필요 인력", "배치 인력", "부족 인원",
    "부족 비용(원)",
    "is_holiday", "is_myeongjeol", "is_free_rule"
]].round(2))

,origin,date,site,total,point_pred,q80,필요 인력,배치 인력,부족 인원,부족 비용(원),is_holiday,is_myeongjeol,is_free_rule
0,2025-10-08,2025-10-08,경복궁,126362.0,59940.878906,73592.828125,63.18,36.799999,26.38,15830751.95,1,1,1
1,2025-10-08,2025-10-09,경복궁,98771.0,42953.550781,55771.281250,49.39,27.889999,21.50,12899917.63,1,1,1
2,2025-04-30,2025-05-04,종묘,44337.0,2983.120117,5033.140137,22.17,2.520000,19.65,11791158.95,0,0,0
3,2025-11-05,2025-11-08,덕수궁,66257.0,29062.259766,35356.500000,33.13,17.680000,15.45,9270151.53,0,0,0
4,2025-10-08,2025-10-08,창경궁,58122.0,20931.359375,27419.630859,29.06,13.710000,15.35,9210710.41,1,1,1
5,2025-10-08,2025-10-08,창덕궁,56508.0,24011.449219,29015.779297,28.25,14.510000,13.75,8247664.43,1,1,1
6,2025-10-29,2025-11-04,창덕궁,45452.0,14675.919922,19185.890625,22.73,9.590000,13.13,7879833.51,0,0,0
7,2025-10-08,2025-10-08,덕수궁,49985.0,21030.519531,24937.619141,24.99,12.470000,12.52,7514215.10,1,1,1
8,2025-10-08,2025-10-09,창경궁,44822.0,16018.950195,20859.140625,22.41,10.430000,11.98,7188857.31,1,1,1
9,2025-10-08,2025-10-09,창덕궁,44893.0,17509.630859,21466.359375,22.45,10.730000,11.71,7027990.83,1,1,1


In [27]:
top10_shortages.to_csv(
    "chronos_calendar_top10_shortages.csv",
    index=False,
    encoding="utf-8-sig"
)
files.download("chronos_calendar_top10_shortages.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 달력 정보 추가 후 오차 분석

### 1. 무엇을 했나
Chronos에 **요일·정기 휴관·공휴일·명절·설/추석 offset·무료일 규칙**을 단계적으로 추가하고, 같은 원점 52개에서 성능과 비용을 비교했다.

현재 달력 모델의 결과:

- **MASE:** 0.822
- **WAPE:** 29.148%
- **적중률:** 81.170%
- **총비용:** 약 8.29억 원

과거 관람객만 쓴 Chronos보다 **약 1억 7,571만 원의 비용을 절감**했다.

### 2. 명절·무료일에서도 개선됐나

| 유형 | 적중률: 기본 → 달력 추가 | 비용 절감 |
|---|---|---:|
| 명절 연휴 | 31.1% → **66.9%** | 약 1억 2,096만 원 |
| 무료일·명절 제외 | 42.8% → **59.8%** | 약 3,607만 원 |

전체 절감액의 약 **89.4%가 명절·무료일에서 발생**했다. 다만 두 유형의 적중률은 여전히 목표 80%보다 낮았다.

### 3. 인력 부족 상위 10건 확인
배치 기준인 **0.8 분위수 예측값**을 사용해 부족 인원을 계산했다.

- 필요 인력 = 실제 관람객 ÷ 2,000
- 배치 인력 = 0.8 분위수 예측 ÷ 2,000
- 부족 인원 = 필요 인력 − 배치 인력, 최소 0명
- 부족 비용 = 부족 인원 × 하루 60만 원

**발견**

- 상위 10건 중 **7건이 2025년 10월 8~9일 명절 연휴**에 집중됐다.
- 최대 부족은 **10월 8일 경복궁 26.38명**, 약 **1,583만 원**이었다.
- **5월 4일 종묘**는 가이드의 종묘대제 날짜와 일치한다.
- **11월 8일 덕수궁·11월 4일 창덕궁**은 원인 확인이 필요하다.

### 4. 저장한 파일
**`chronos_calendar_top10_shortages.csv`**

현재 달력 모델에서 **부족 인원이 가장 큰 개소×날짜 10건**을 저장한 파일이다. 원점·날짜·개소·실제 관람객·점예측·0.8 분위수·필요/배치/부족 인력·부족 비용·달력 플래그가 포함된다.

**용도:** 공식 공지로 원인을 확인하고, 보정 규칙을 정하는 데 사용한다. 보고서의 ‘인력 부족 상위 10건’ 표를 작성하는 기초 자료다.

※ 전체 예측이나 최종 2026년 권고표를 저장한 파일은 아니다.

### 5. 다음 단계
- 공식 공지로 원인과 공지 시점을 확인한다.
- 근무표 확정 전에 알 수 있었던 정보만으로 보정 규칙을 만든다.
- 같은 원점에서 보정 전후 비용을 비교한다.
- 최종 모델이 바뀌면 상위 10건도 다시 계산한다.

**현재는 오류를 찾은 단계이며, 원인 확인·보정 실험은 아직 완료하지 않았다.**

In [28]:
RAIN_THRESHOLD = 10.0

rain_history = model_panel[
    ["site", "date", "precip_mm"]
].copy()

# 과거 날짜마다 그 전날의 강수량을 정확한 날짜로 연결
rain_history["date"] += pd.Timedelta(days=1)
rain_history = rain_history.rename(
    columns={"precip_mm": "previous_day_precip"}
)

weather_panel = free_panel.merge(
    rain_history,
    on=["site", "date"],
    how="left",
    validate="one_to_one"
)

weather_panel["previous_day_heavy_rain"] = (
    weather_panel["previous_day_precip"]
    .ge(RAIN_THRESHOLD)
    .astype(float)
)

# 전날 기록이 없으면 '비 안 옴'으로 단정하지 않기
weather_panel.loc[
    weather_panel["previous_day_precip"].isna(),
    "previous_day_heavy_rain"
] = np.nan

weather_panel = weather_panel.drop(
    columns="previous_day_precip"
)

weather_features = [
    *free_features, "previous_day_heavy_rain"
]

def make_weather_inputs(origin):
    context = weather_panel.loc[
        weather_panel["date"] < origin
    ].copy()

    future = weather_panel.loc[
        weather_panel["date"].between(
            origin,
            origin + pd.Timedelta(days=HORIZON - 1)
        ),
        ["site", "date", *free_features]
    ].copy()

    last_weather = model_panel.loc[
        model_panel["date"].eq(
            origin - pd.Timedelta(days=1)
        ),
        ["site", "precip_mm"]
    ].copy()

    assert last_weather["precip_mm"].notna().all()
    assert last_weather["site"].nunique() == 23

    last_weather["previous_day_heavy_rain"] = (
        last_weather["precip_mm"]
        .ge(RAIN_THRESHOLD).astype(float)
    )

    future = future.merge(
        last_weather[["site", "previous_day_heavy_rain"]],
        on="site",
        how="left",
        validate="many_to_one"
    )

    return context, future

context, future = make_weather_inputs(origins[0])

print("未来入力行数:", len(future))
print("雨変数欠損:", future["previous_day_heavy_rain"].isna().sum())

未来入力行数: 161
雨変数欠損: 0


In [29]:
weather_dir = Path("outputs/chronos2_past_rain")
weather_dir.mkdir(parents=True, exist_ok=True)

weather_results = []

for origin in tqdm(origins, desc="원점 직전 큰비 정보 추가"):
    checkpoint = weather_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        context, future = make_weather_inputs(origin)

        raw = pipeline.predict_df(
            df=context,
            future_df=future,
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=HORIZON,
            quantile_levels=[0.5, 0.8],
            cross_learning=False,
            freq="D"
        )

        result = format_calendar_result(raw, origin)
        result["model"] = "chronos2_calendar_origin_rain"
        result.to_csv(checkpoint, index=False)

    weather_results.append(result)

chronos_weather_predictions = pd.concat(
    weather_results, ignore_index=True
)

assert len(chronos_weather_predictions) == 8372
assert not chronos_weather_predictions.duplicated(
    ["origin", "site", "date"]
).any()

weather_eval = chronos_weather_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left", validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left", validate="many_to_one"
)

weather_summary = pd.DataFrame([
    evaluate_predictions(
        weather_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos 달력 + 원점 직전 큰비(7일 유지)"
    )
])

display(pd.concat(
    [free_summary, weather_summary],
    ignore_index=True
).round(3))

원점 직전 큰비 정보 추가:   0%|          | 0/52 [00:00<?, ?it/s]

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,Chronos + 요일·휴관 + 공휴일·명절 + offset + 무료일,0.822,29.148,81.170,4.383769e+08,3.901462e+08,8.285232e+08
1,Chronos 달력 + 원점 직전 큰비(7일 유지),0.821,29.075,81.323,4.349492e+08,3.922506e+08,8.271998e+08


In [30]:
perfect_panel = weather_panel.merge(
    model_panel[["date", "site", "precip_mm"]],
    on=["date", "site"],
    how="left",
    validate="one_to_one"
)

assert perfect_panel["precip_mm"].notna().all()

perfect_panel["today_heavy_rain"] = (
    perfect_panel["precip_mm"].ge(10).astype(float)
)
perfect_panel = perfect_panel.drop(columns="precip_mm")

perfect_dir = Path("outputs/chronos2_perfect_rain")
perfect_dir.mkdir(parents=True, exist_ok=True)

perfect_results = []

for origin in tqdm(origins, desc="완벽한 예보 가정"):
    checkpoint = perfect_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        context = perfect_panel.loc[
            perfect_panel["date"] < origin
        ].copy()

        # 앞 실험과 같은 원점 직전 큰비 변수 유지
        _, future = make_weather_inputs(origin)

        # 미래 당일 실측: 완벽한 예보 가정에서만 사용
        future = future.merge(
            perfect_panel[["site", "date", "today_heavy_rain"]],
            on=["site", "date"],
            how="left",
            validate="one_to_one"
        )

        raw = pipeline.predict_df(
            df=context,
            future_df=future,
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=HORIZON,
            quantile_levels=[0.5, 0.8],
            cross_learning=False,
            freq="D"
        )

        result = format_calendar_result(raw, origin)
        result["model"] = "chronos2_perfect_rain_assumption"
        result.to_csv(checkpoint, index=False)

    perfect_results.append(result)

chronos_perfect_predictions = pd.concat(
    perfect_results, ignore_index=True
)

assert len(chronos_perfect_predictions) == 8372
assert not chronos_perfect_predictions.duplicated(
    ["origin", "site", "date"]
).any()

perfect_eval = chronos_perfect_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left", validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left", validate="many_to_one"
)

perfect_summary = pd.DataFrame([
    evaluate_predictions(
        perfect_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos + 当日 큰비 [완벽한 예보 가정]"
    )
])

display(pd.concat(
    [free_summary, weather_summary, perfect_summary],
    ignore_index=True
).round(3))

완벽한 예보 가정:   0%|          | 0/52 [00:00<?, ?it/s]

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,Chronos + 요일·휴관 + 공휴일·명절 + offset + 무료일,0.822,29.148,81.170,4.383769e+08,3.901462e+08,8.285232e+08
1,Chronos 달력 + 원점 직전 큰비(7일 유지),0.821,29.075,81.323,4.349492e+08,3.922506e+08,8.271998e+08
2,Chronos + 当日 큰비 [완벽한 예보 가정],0.813,28.800,81.351,4.283073e+08,3.922664e+08,8.205737e+08


## 🌧️ 날씨 정보 추가 실험 결과

### 실험 목적과 조건

달력 정보를 넣은 Chronos에 날씨를 추가했을 때, 예측 오차와 인력 배치 비용이 줄어드는지 확인했다.

- **학습 기록:** 2023년부터 각 원점 전날까지
- **평가:** 2025년 원점 52개, 원점마다 7일 예측
- **평가 날짜:** 2025-01-01~12-30
- **큰비 기준:** 하루 강수량 **10mm 이상**
- **정확도:** 0.5 분위수로 MASE·WAPE 계산
- **배치:** 0.8 분위수로 적중률·비용 계산

### 비교한 세 가지 설정

| 설정 | 사용한 날씨 정보 | 운영 적용 |
|---|---|---|
| 달력 모델 | 날씨 없음 | 가능 |
| 과거 날씨 추가 | 원점 전날의 큰비 여부를 예측 7일 동안 유지 | 해당 가정으로 가능 |
| 완벽한 예보 가정 | 위 정보 + 예측할 날의 실제 큰비 여부 | **참고 실험** |

**주의:** 과거 날씨 실험은 매 예측일의 전날 날씨를 안다는 뜻이 아니다. 근무표 확정 시점에 아는 **원점 전날의 정보만 미래 7일에 고정**했다.

### 결과

| 설정 | MASE ↓ | WAPE ↓ | 적중률 | 총비용 |
|---|---:|---:|---:|---:|
| 달력 모델 | 0.822 | 29.148% | 81.170% | 약 8.285억 원 |
| + 원점 직전 큰비 | 0.821 | 29.075% | 81.323% | 약 8.272억 원 |
| + 당일 큰비 **[완벽한 예보 가정]** | **0.813** | **28.800%** | **81.351%** | **약 8.206억 원** |

### 해석

- 과거 날씨 추가로 **약 132만 원, 0.16%**의 비용이 줄었다.
- 당일 큰비 정보까지 정확히 안다고 가정하면, 과거 날씨 모델보다 **약 663만 원, 0.8%**가 더 줄었다.
- 이번 설정에서는 **날씨보다 공휴일·명절·무료일 정보의 비용 개선 폭이 컸다.**
- 큰비 여부만 사용했으므로, 날씨 정보 전체가 중요하지 않다고 단정하지 않는다.
- 완벽한 예보 결과는 실제 운영 결과와 구분한다. **2026년 권고표에는 미래 실측 날씨를 그대로 사용할 수 없다.**
- 날씨 피처의 최종 채택 여부는 아직 확정하지 않았다.

## 📝 기존 출력 문구 정정

일부 코드의 출력 문구와 모델 이름에 일본어가 섞였다. **출력용 문자열의 오류이며, 그 문구 자체가 계산 결과를 바꾸지는 않았다.**

| 기존 문구 | 한국어 의미·수정 이름 |
|---|---|
| `予測 행 수` | 예측 행 수 |
| `MASE分母` | MASE 분모 |
| `評価準備行数` | 평가 준비 행 수 |
| `1,196件・8,372行` | 1,196건·8,372행 |
| `曜日` | 요일 |
| `不足` | 부족 |
| `未来入力行数` | 미래 입력 행 수 |
| `雨変数欠損` | 비 변수 결측 |
| `当日 큰비` | 당일 큰비 |

앞으로 코드 출력과 저장용 표 이름은 **한국어로 통일**한다.

In [31]:
import json
import shutil
from pathlib import Path
from importlib.metadata import version

save_dir = Path("outputs/chronos_fva_2025_52origins")
save_dir.mkdir(parents=True, exist_ok=True)

# 예측 결과 7개 모델
all_chronos_predictions = pd.concat([
    chronos_predictions,
    chronos_calendar_predictions,
    chronos_holiday_predictions,
    chronos_offset_predictions,
    chronos_free_predictions,
    chronos_weather_predictions,
    chronos_perfect_predictions
], ignore_index=True)

assert len(all_chronos_predictions) == 7 * 52 * 23 * 7
assert not all_chronos_predictions.duplicated(
    ["origin", "site", "date", "model"]
).any()

# 한국어 이름으로 평가표 정리
fva_saved = pd.concat([
    chronos_summary, calendar_summary, holiday_summary,
    offset_summary, free_summary,
    weather_summary, perfect_summary
], ignore_index=True)

fva_saved["방법"] = [
    "Chronos: 과거 관람객만",
    "+ 요일·정기 휴관",
    "+ 공휴일·명절",
    "+ 설·추석 offset 및 근처 표시",
    "+ 무료 개방일 규칙",
    "+ 원점 직전 큰비(7일 유지)",
    "+ 당일 큰비 [완벽한 예보 가정]"
]

fva_saved["앞 행 대비 절감액(원)"] = (
    fva_saved["총비용(원)"].shift()
    - fva_saved["총비용(원)"]
)

fva_saved["완벽한 예보 가정"] = [False] * 6 + [True]

tables = {
    "chronos_predictions.csv": all_chronos_predictions,
    "chronos_fva_summary.csv": fva_saved,
    "date_type_comparison.csv": type_comparison,
    "calendar_model_top10_shortages.csv": top10_shortages,
    "mase_scales.csv": scale_table
}

for filename, table in tables.items():
    table.to_csv(
        save_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

settings = {
    "model": "amazon/chronos-2",
    "chronos_version": version("chronos-forecasting"),
    "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "history_start": "2023-01-01",
    "origins": [str(o.date()) for o in origins],
    "horizon_days": HORIZON,
    "point_prediction": "0.5 분위수",
    "staffing_prediction": "0.8 분위수",
    "evaluation_rule": "실제 total > 0인 날",
    "mase_scale": "원점 전까지 두 y가 유효한 7일 차분의 평균",
    "mase_aggregation": "개소×원점별 MASE의 동일 가중 평균",
    "visitors_per_staff": 2000,
    "shortage_cost_per_staff": 600000,
    "surplus_cost_per_staff": 150000,
    "rain_threshold_mm": 10,
    "future_past_rain_rule": "원점 전날 큰비 여부를 7일 유지",
    "cross_learning": False,
    "context_length_setting": "모델 기본값",
    "model_context_length": getattr(
        pipeline, "context_length", None
    ),
    "note": (
        "유형별 비교와 상위 10건은 날씨 없는 달력 모델 기준. "
        "최종 모델 선택 시 재계산 필요."
    )
}

with open(save_dir / "settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, ensure_ascii=False, indent=2, default=str)

archive = shutil.make_archive(
    "chronos_fva_2025_52origins",
    "zip",
    root_dir=save_dir
)

print("저장한 예측 행 수:", len(all_chronos_predictions))
print("저장 파일 수:", len(list(save_dir.iterdir())))
files.download(archive)

저장한 예측 행 수: 58604
저장 파일 수: 6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
# 기존 데이터·무료일 규칙은 보존
notice_rules = pd.DataFrame([
    {
        "name": "종묘대제",
        "published": "2025-04-28",
        "start": "2025-05-04",
        "end": "2025-05-04",
        "site": "종묘"
    },
    {
        "name": "코리아 그랜드 페스티벌",
        "published": "2025-10-16",
        "start": "2025-10-29",
        "end": "2025-11-09",
        "site": "전체"
    }
])

for col in ["published", "start", "end"]:
    notice_rules[col] = pd.to_datetime(notice_rules[col])

def make_notice_inputs(origin):
    # 날씨 없는 달력 모델을 기준으로 보정 효과 비교
    panel = free_panel.copy()
    cutoff = origin - pd.Timedelta(days=1)

    # 해당 원점의 근무표 확정 전에 공개된 공지만 반영
    known = notice_rules.loc[
        notice_rules["published"] <= cutoff
    ]

    for rule in known.itertuples():
        mask = panel["date"].between(rule.start, rule.end)

        if rule.site != "전체":
            mask &= panel["site"].eq(rule.site)

        panel.loc[mask, "is_free_rule"] = 1

    context = panel.loc[panel["date"] < origin].copy()

    future = panel.loc[
        panel["date"].between(
            origin,
            origin + pd.Timedelta(days=HORIZON - 1)
        ),
        ["site", "date", *free_features]
    ].copy()

    return context, future

# 종묘대제가 포함된 원점에서 입력 확인
context, future = make_notice_inputs(pd.Timestamp("2025-04-30"))

display(future.loc[
    future["site"].eq("종묘"),
    ["site", "date", "is_free_rule"]
])

,site,date,is_free_rule
17290,종묘,2025-04-30,1
17291,종묘,2025-05-01,0
17292,종묘,2025-05-02,0
17293,종묘,2025-05-03,0
17294,종묘,2025-05-04,1
17295,종묘,2025-05-05,0
17296,종묘,2025-05-06,0


In [33]:
notice_dir = Path("outputs/chronos2_notice_correction")
notice_dir.mkdir(parents=True, exist_ok=True)

notice_results = []

for origin in tqdm(origins, desc="사전 공지 무료일 보정"):
    checkpoint = notice_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        context, future = make_notice_inputs(origin)

        raw = pipeline.predict_df(
            df=context,
            future_df=future,
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=HORIZON,
            quantile_levels=[0.5, 0.8],
            cross_learning=False,
            freq="D"
        )

        result = format_calendar_result(raw, origin)
        result["model"] = "chronos2_calendar_notice_correction"
        result.to_csv(checkpoint, index=False)

    notice_results.append(result)

chronos_notice_predictions = pd.concat(
    notice_results, ignore_index=True
)

assert len(chronos_notice_predictions) == 8372
assert not chronos_notice_predictions.duplicated(
    ["origin", "site", "date"]
).any()

notice_eval = chronos_notice_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left",
    validate="many_to_one"
)

notice_summary = pd.DataFrame([
    evaluate_predictions(
        notice_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="Chronos 달력 + 사전 공지 무료일 보정"
    )
])

notice_comparison = pd.concat(
    [free_summary, notice_summary],
    ignore_index=True
)

notice_comparison["앞 행 대비 절감액(원)"] = (
    notice_comparison["총비용(원)"].shift()
    - notice_comparison["총비용(원)"]
)

display(notice_comparison.round(3))

사전 공지 무료일 보정:   0%|          | 0/52 [00:00<?, ?it/s]

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),앞 행 대비 절감액(원)
0,Chronos + 요일·휴관 + 공휴일·명절 + offset + 무료일,0.822,29.148,81.170,4.383769e+08,3.901462e+08,8.285232e+08,NaN
1,Chronos 달력 + 사전 공지 무료일 보정,0.802,28.456,82.159,3.382963e+08,4.461166e+08,7.844128e+08,4.411032e+07


In [34]:
before_after_top10 = top10_shortages[[
    "origin", "site", "date", "total", "q80", "부족 비용(원)"
]].rename(columns={
    "q80": "보정 전 q80",
    "부족 비용(원)": "보정 전 부족 비용"
}).merge(
    notice_eval[["origin", "site", "date", "q80"]],
    on=["origin", "site", "date"],
    how="left",
    validate="one_to_one"
).rename(columns={"q80": "보정 후 q80"})

before_after_top10["보정 후 부족 비용"] = (
    (before_after_top10["total"]
     - before_after_top10["보정 후 q80"])
    .clip(lower=0) / 2000 * 600_000
)

display(before_after_top10.round(2))

,origin,site,date,total,보정 전 q80,보정 전 부족 비용,보정 후 q80,보정 후 부족 비용
0,2025-10-08,경복궁,2025-10-08,126362.0,73592.828125,15830751.95,73592.828125,15830751.56
1,2025-10-08,경복궁,2025-10-09,98771.0,55771.281250,12899917.63,55771.281250,12899917.97
2,2025-04-30,종묘,2025-05-04,44337.0,5033.140137,11791158.95,7836.819824,10950053.91
3,2025-11-05,덕수궁,2025-11-08,66257.0,35356.500000,9270151.53,42124.671875,7239698.44
4,2025-10-08,창경궁,2025-10-08,58122.0,27419.630859,9210710.41,27419.630859,9210710.16
5,2025-10-08,창덕궁,2025-10-08,56508.0,29015.779297,8247664.43,29015.779297,8247664.45
6,2025-10-29,창덕궁,2025-11-04,45452.0,19185.890625,7879833.51,43655.859375,538842.19
7,2025-10-08,덕수궁,2025-10-08,49985.0,24937.619141,7514215.10,24937.619141,7514214.84
8,2025-10-08,창경궁,2025-10-09,44822.0,20859.140625,7188857.31,20859.140625,7188857.23
9,2025-10-08,창덕궁,2025-10-09,44893.0,21466.359375,7027990.83,21466.359375,7027991.02


In [35]:
for label in ["전", "후"]:
    before_after_top10[f"보정 {label} 부족 비용"] = (
        before_after_top10["total"].astype("float64")
        - before_after_top10[f"보정 {label} q80"].astype("float64")
    ).clip(lower=0) / 2000 * 600_000

before_after_top10["부족 비용 절감액"] = (
    before_after_top10["보정 전 부족 비용"]
    - before_after_top10["보정 후 부족 비용"]
)

display(before_after_top10.round(2))

,origin,site,date,total,보정 전 q80,보정 전 부족 비용,보정 후 q80,보정 후 부족 비용,부족 비용 절감액
0,2025-10-08,경복궁,2025-10-08,126362.0,73592.828125,15830751.56,73592.828125,15830751.56,0.00
1,2025-10-08,경복궁,2025-10-09,98771.0,55771.281250,12899917.97,55771.281250,12899917.97,0.00
2,2025-04-30,종묘,2025-05-04,44337.0,5033.140137,11791158.98,7836.819824,10950053.91,841105.08
3,2025-11-05,덕수궁,2025-11-08,66257.0,35356.500000,9270151.17,42124.671875,7239698.44,2030452.73
4,2025-10-08,창경궁,2025-10-08,58122.0,27419.630859,9210710.16,27419.630859,9210710.16,0.00
5,2025-10-08,창덕궁,2025-10-08,56508.0,29015.779297,8247664.45,29015.779297,8247664.45,0.00
6,2025-10-29,창덕궁,2025-11-04,45452.0,19185.890625,7879833.40,43655.859375,538842.19,7340991.21
7,2025-10-08,덕수궁,2025-10-08,49985.0,24937.619141,7514214.84,24937.619141,7514214.84,0.00
8,2025-10-08,창경궁,2025-10-09,44822.0,20859.140625,7188857.23,20859.140625,7188857.23,0.00
9,2025-10-08,창덕궁,2025-10-09,44893.0,21466.359375,7027991.02,21466.359375,7027991.02,0.00


In [36]:
notice_shortages = notice_eval.loc[
    notice_eval["total"].gt(0)
].copy()

notice_shortages["부족 인원"] = (
    notice_shortages["total"].astype("float64")
    - notice_shortages["q80"].astype("float64")
).clip(lower=0) / 2000

notice_shortages["부족 비용(원)"] = (
    notice_shortages["부족 인원"] * 600_000
)

notice_top10 = notice_shortages.nlargest(10, "부족 인원")

display(notice_top10[[
    "origin", "date", "site", "total",
    "point_pred", "q80", "부족 인원", "부족 비용(원)"
]].round(2))

,origin,date,site,total,point_pred,q80,부족 인원,부족 비용(원)
6440,2025-10-08,2025-10-08,경복궁,126362.0,59940.878906,73592.828125,26.38,15830751.56
6441,2025-10-08,2025-10-09,경복궁,98771.0,42953.550781,55771.281250,21.50,12899917.97
2846,2025-04-30,2025-05-04,종묘,44337.0,5329.290039,7836.819824,18.25,10950053.91
6552,2025-10-08,2025-10-08,창경궁,58122.0,20931.359375,27419.630859,15.35,9210710.16
6559,2025-10-08,2025-10-08,창덕궁,56508.0,24011.449219,29015.779297,13.75,8247664.45
6461,2025-10-08,2025-10-08,덕수궁,49985.0,21030.519531,24937.619141,12.52,7514214.84
7108,2025-11-05,2025-11-08,덕수궁,66257.0,34352.261719,42124.671875,12.07,7239698.44
6553,2025-10-08,2025-10-09,창경궁,44822.0,16018.950195,20859.140625,11.98,7188857.23
6560,2025-10-08,2025-10-09,창덕궁,44893.0,17509.630859,21466.359375,11.71,7027991.02
6545,2025-10-08,2025-10-08,종묘,35524.0,9863.660156,13771.540039,10.88,6525736.52


In [37]:
notice_save_dir = Path("outputs/chronos_notice_correction")
notice_save_dir.mkdir(parents=True, exist_ok=True)

notice_tables = {
    "predictions.csv": chronos_notice_predictions,
    "comparison_summary.csv": notice_comparison,
    "original_top10_before_after.csv": before_after_top10,
    "corrected_top10.csv": notice_top10,
    "notice_rules.csv": notice_rules
}

for filename, table in notice_tables.items():
    table.to_csv(
        notice_save_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

archive = shutil.make_archive(
    "chronos_notice_correction_2025_52origins",
    "zip",
    root_dir=notice_save_dir
)

files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📢 사전 공지 반영 · 인력 부족 보정 실험

### 1. 실험 목적

기존 달력 모델은 **명절·마지막 수요일만 무료일로 표시**했다. 달력에 없는 행사·임시 무료 개방을 놓친 것이 큰 오차와 관련 있는지 확인하고, **사전 공지를 반영했을 때 비용이 줄어드는지** 비교했다.

### 2. 추가한 정보

궁능유적본부 공식 공지에서 확인한 두 일정을 기존 무료일 변수에 추가했다.

| 공지 | 적용 날짜·대상 | 게시일 |
|---|---|---|
| 종묘대제 무료관람 | 2025-05-04 · 종묘 | 2025-04-28 |
| 코리아 그랜드 페스티벌 무료 개방 | 2025-10-29~11-09 · 4대궁·종묘·왕릉·세종유적 | 2025-10-16 |

**각 원점의 근무표 확정일인 원점 전날까지 공개된 공지만 반영했다.** 미래 실제 관람객이나 행사 당일 관람객 배수는 사용하지 않았다.

※ 무료일 표시는 관람 수요를 위한 정보이며, 휴관일 개방을 뜻하지 않는다. 창덕궁 후원 특별관람은 무료 개방 대상에서 제외된다.

### 3. 비교 조건

- **기준 모델:** Chronos + 요일·정기 휴관·공휴일·명절·offset·무료일 규칙
- **보정:** 같은 모델의 무료일 표시에 사전 공지 일정 추가
- **날씨:** 사용하지 않음
- **과거 입력:** 2023년부터 각 원점 전날까지
- **백테스트:** 2025년 원점 52개, 원점마다 7일 예측
- **평가 날짜:** 2025-01-01~12-30 중 실제 관람객이 있는 날
- **정확도:** 0.5 분위수 / **배치·비용:** 0.8 분위수

### 4. 전체 결과

| 항목 | 보정 전 | 보정 후 |
|---|---:|---:|
| MASE ↓ | 0.822 | **0.802** |
| WAPE ↓ | 29.148% | **28.456%** |
| 적중률 | 81.170% | **82.159%** |
| 부족 비용 | 약 4.38억 원 | 약 3.38억 원 |
| 과잉 비용 | 약 3.90억 원 | 약 4.46억 원 |
| 총비용 | 약 8.29억 원 | **약 7.84억 원** |

**보정 전 대비 총비용 약 4,411만 원, 5.3% 절감**

부족 비용이 약 1억 8만 원 감소하고 과잉 비용이 약 5,597만 원 증가해, 총비용은 줄었다.

### 5. 기존 부족 상위 10건의 변화

| 사례 | 부족 비용: 보정 전 → 후 | 절감액 |
|---|---|---:|
| 5/4 종묘 | 약 1,179만 → 1,095만 원 | 약 84만 원 |
| 11/8 덕수궁 | 약 927만 → 724만 원 | 약 203만 원 |
| 11/4 창덕궁 | 약 788만 → **54만 원** | 약 734만 원 |

기존 상위 10건의 **부족 비용 합계는 약 1,021만 원 감소**했다. 추석 7건은 이미 무료일로 표시돼 있어 이번 보정으로 예측이 변하지 않았다.

※ 전체 절감액 4,411만 원은 모든 평가 날짜의 부족·과잉 비용을 합친 값이므로, 위 10건의 절감액과 다르다.

### 6. 보정 후 남은 문제

새로운 부족 상위 10건은 **추석 연휴 8건·종묘대제 1건·페스티벌 1건**이었다.

- 11/4 창덕궁은 상위 10건에서 제외됐다.
- 최대 부족은 여전히 **10/8 경복궁 26.38명**이었다.
- 무료일 표시를 보완해도 **명절·큰 행사에서 급증 규모를 충분히 예측하지 못하는 문제**는 남았다.

**해석:** 사전 공지 반영은 일부 행사일의 부족 비용을 줄였다. 다만 큰 오차를 확인한 뒤 만든 규칙이므로 **탐색적 보정 결과**이며, 독립적인 추가 검증 없이 일반적인 개선 효과로 확정하지 않는다.

### 7. 저장 파일과 공유 용도

**압축 파일:** `chronos_notice_correction_2025_52origins.zip`

| 파일 | 내용·용도 |
|---|---|
| `predictions.csv` | 보정 모델의 전체 예측 8,372행. 평가 재사용·모델 비교 |
| `comparison_summary.csv` | 보정 전후 MASE·WAPE·적중률·비용 비교 |
| `original_top10_before_after.csv` | 기존 부족 상위 10건의 보정 전후 변화 |
| `corrected_top10.csv` | 보정 후 새로 계산한 부족 상위 10건 |
| `notice_rules.csv` | 추가한 공지의 행사명·게시일·적용 기간·대상 |

**공유 목적:** 사전 공지 정보를 반영한 방법, 비용 변화, 남은 오류를 팀원이 확인하고 보고서 근거로 활용할 수 있게 저장했다.

**남은 정리:** `notice_rules.csv`에 공식 공지 주소를 추가해야 한다.

### 공식 공지 주소

- [종묘대제 봉행 안내](https://royal.khs.go.kr/ROYAL/contents/R403000000.do?id=20250428101959471503&schM=view)
- [코리아 그랜드 페스티벌 무료입장 안내](https://royal.khs.go.kr/ROYAL/contents/R403000000.do?id=20251016153922339892&schM=view)

In [38]:
notice_by_group = notice_eval.merge(
    model_panel[["date", "site", "group"]],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
)

assert notice_by_group["group"].notna().all()

group_results = []

for group, data in notice_by_group.groupby("group"):
    result = evaluate_predictions(
        data,
        accuracy_col="point_pred",
        staffing_col="q80",
        name=group
    )

    evaluated = data.loc[data["total"].gt(0)]
    result["개소 수"] = evaluated["site"].nunique()
    result["평가건수"] = len(evaluated)
    result["건당 비용(원)"] = (
        result["총비용(원)"] / len(evaluated)
    )
    group_results.append(result)

notice_group_summary = pd.DataFrame(group_results)
display(notice_group_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),개소 수,평가건수,건당 비용(원)
0,궁,0.785,26.665,81.066,2.404256e+08,3.347106e+08,5.751362e+08,4,1257,457546.685
1,왕릉,0.782,33.689,82.478,5.957211e+07,8.465996e+07,1.442321e+08,18,5610,25709.816
2,종묘,1.247,42.839,80.831,3.829859e+07,2.674600e+07,6.504459e+07,1,313,207810.177


In [39]:
notice_by_weekday = notice_eval.copy()
notice_by_weekday["날짜 유형"] = np.where(
    notice_by_weekday["date"].dt.dayofweek < 5,
    "평일", "주말"
)

weekday_results = []

for label, data in notice_by_weekday.groupby("날짜 유형"):
    result = evaluate_predictions(
        data,
        accuracy_col="point_pred",
        staffing_col="q80",
        name=label
    )
    result["평가건수"] = data["total"].gt(0).sum()
    weekday_results.append(result)

notice_weekday_summary = pd.DataFrame(weekday_results)
display(notice_weekday_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),평가건수
0,주말,1.021,28.079,82.419,1.039506e+08,1.736814e+08,2.776321e+08,2389
1,평일,0.704,28.691,82.029,2.343456e+08,2.724351e+08,5.067808e+08,4791


In [40]:
notice_by_type = notice_eval.merge(
    chronos_by_type[["origin", "site", "date", "date_type"]],
    on=["origin", "site", "date"],
    how="left",
    validate="one_to_one"
)

assert notice_by_type["date_type"].notna().all()

notice_type_results = []

for label, data in notice_by_type.groupby("date_type"):
    result = evaluate_predictions(
        data,
        accuracy_col="point_pred",
        staffing_col="q80",
        name=label
    )
    result["평가건수"] = data["total"].gt(0).sum()
    notice_type_results.append(result)

notice_type_summary = pd.DataFrame(notice_type_results)
display(notice_type_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),평가건수
0,공휴일(명절·무료일 제외),1.828,37.023,37.198,3.391313e+07,4.338954e+06,3.825209e+07,207
1,그 외 개방일,0.666,25.464,84.919,1.629933e+08,3.717440e+08,5.347373e+08,6445
2,명절 연휴,3.627,47.851,67.224,1.136532e+08,5.323866e+07,1.668918e+08,299
3,무료 개방일(명절 제외),0.970,27.573,64.629,2.773667e+07,1.679499e+07,4.453166e+07,229


In [41]:
notice_by_type_updated = notice_by_type.copy()
notice_by_type_updated["공지 무료일"] = False

for rule in notice_rules.itertuples():
    mask = (
        notice_by_type_updated["date"].between(rule.start, rule.end)
        & (
            notice_by_type_updated["origin"]
            - pd.Timedelta(days=1)
        ).ge(rule.published)
    )

    if rule.site != "전체":
        mask &= notice_by_type_updated["site"].eq(rule.site)

    notice_by_type_updated.loc[mask, "공지 무료일"] = True

notice_by_type_updated.loc[
    notice_by_type_updated["공지 무료일"]
    & notice_by_type_updated["date_type"].ne("명절 연휴"),
    "date_type"
] = "무료 개방일(명절 제외)"

updated_results = []

for label, data in notice_by_type_updated.groupby("date_type"):
    result = evaluate_predictions(
        data,
        accuracy_col="point_pred",
        staffing_col="q80",
        name=label
    )
    result["평가건수"] = data["total"].gt(0).sum()
    updated_results.append(result)

notice_type_updated_summary = pd.DataFrame(updated_results)
display(notice_type_updated_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),평가건수
0,공휴일(명절·무료일 제외),1.828,37.023,37.198,3.391313e+07,4.338954e+06,3.825209e+07,207
1,그 외 개방일,0.614,23.892,85.130,1.232613e+08,3.064774e+08,4.297388e+08,6214
2,명절 연휴,3.627,47.851,67.224,1.136532e+08,5.323866e+07,1.668918e+08,299
3,무료 개방일(명절 제외),1.217,34.022,71.957,6.746861e+07,8.206154e+07,1.495301e+08,460


In [42]:
type_save_dir = Path("outputs/chronos_type_evaluation")
type_save_dir.mkdir(parents=True, exist_ok=True)

type_tables = {
    "by_site_group.csv": notice_group_summary,
    "by_weekday_weekend.csv": notice_weekday_summary,
    "by_date_type_original_rule.csv": notice_type_summary,
    "by_date_type_with_notices.csv": notice_type_updated_summary
}

for filename, table in type_tables.items():
    table.to_csv(
        type_save_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

archive = shutil.make_archive(
    "chronos_type_evaluation_2025_52origins",
    "zip",
    root_dir=type_save_dir
)

files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📊 공지 보정 모델 · 유형별 평가

### 1. 무엇을 확인했나?

전체 평균에 가려지는 오류를 찾기 위해 **개소 유형·평일/주말·명절/무료일/공휴일별로 성능과 비용을 나눠 평가**했다.

- **대상 모델:** 달력 정보 + 사전 공지 무료일을 반영한 Chronos
- **평가:** 2025년 원점 52개, 원점마다 7일 예측
- **평가 날짜:** 2025-01-01~12-30 중 실제 관람객이 있는 날
- **정확도:** 0.5 분위수 / **배치·비용:** 0.8 분위수
- 유형별로 다른 모델을 선택한 것은 아니며, **현재 모델의 취약한 조건을 확인한 단계**다.

### 2. 개소 유형별 결과

| 유형 | 개소 수 | MASE ↓ | WAPE ↓ | 적중률 | 총비용 |
|---|---:|---:|---:|---:|---:|
| 궁 | 4 | 0.785 | 26.665% | 81.066% | 약 5.75억 원 |
| 왕릉 | 18 | 0.782 | 33.689% | 82.478% | 약 1.44억 원 |
| 종묘 | 1 | **1.247** | 42.839% | 80.831% | 약 0.65억 원 |

**발견:** 종묘의 MASE가 가장 높았다. 적중률은 약 80.8%지만, 종묘대제처럼 크게 부족한 날이 있어 **적중률과 부족 규모를 함께 확인해야 한다.**

※ 개소 수·관람객 규모가 다르므로 총비용만으로 유형별 예측 난이도를 비교하지 않는다.

### 3. 평일·주말 결과

| 유형 | MASE ↓ | WAPE ↓ | 적중률 | 평가건수 |
|---|---:|---:|---:|---:|
| 평일 | **0.704** | 28.691% | 82.029% | 4,791 |
| 주말 | 1.021 | **28.079%** | 82.419% | 2,389 |

**발견:** 주말 MASE는 더 높지만, WAPE는 비슷하고 적중률은 모두 약 82%였다. 주말이 모든 기준에서 나쁘다고 해석하지 않는다.

※ 평일·주말에는 공휴일·명절도 포함된다.

### 4. 사전 공지를 포함한 날짜 유형별 결과

기존 달력 분류에 종묘대제·페스티벌 무료 개방을 추가했다. **해당 원점 전날까지 공개된 공지만 반영**했으며, 명절을 우선 분류해 유형이 겹치지 않도록 했다.

| 유형 | MASE ↓ | WAPE ↓ | 적중률 | 평가건수 |
|---|---:|---:|---:|---:|
| 그 외 개방일 | **0.614** | 23.892% | **85.130%** | 6,214 |
| 무료 개방일·명절 제외 | 1.217 | 34.022% | 71.957% | 460 |
| 공휴일·명절/무료일 제외 | 1.828 | 37.023% | **37.198%** | 207 |
| 명절 연휴 | **3.627** | 47.851% | 67.224% | 299 |

**발견**

- 전체 적중률은 82.2%지만, **명절·무료일·나머지 공휴일에서는 목표 80%보다 낮았다.**
- 특히 공휴일 적중률이 낮아 추가 모델 비교와 보정 검토가 필요하다.
- ‘그 외 개방일’에는 일반 평일과 주말이 모두 포함된다.

### 5. 무료일 분류 변경의 의미

| 항목 | 기존 달력 분류 | 공지 포함 분류 |
|---|---:|---:|
| 무료일 평가건수 | 229 | **460** |
| 무료일 MASE | 0.970 | 1.217 |
| 무료일 적중률 | 64.629% | 71.957% |

**예측값은 바꾸지 않고 평가 그룹만 변경했다.** 행사 날짜가 무료일 그룹에 추가되면서 유형별 수치가 달라졌으며, 전체 MASE·총비용은 동일하다.

### 6. 저장 파일

**`chronos_type_evaluation_2025_52origins.zip`**

| 파일 | 내용 |
|---|---|
| `by_site_group.csv` | 궁·종묘·왕릉별 평가 |
| `by_weekday_weekend.csv` | 평일·주말별 평가 |
| `by_date_type_original_rule.csv` | 기존 달력 규칙에 따른 날짜 유형별 평가 |
| `by_date_type_with_notices.csv` | 사전 공지를 포함한 날짜 유형별 평가 |

**공유 용도:** 보고서의 유형별 오차 분석 근거로 활용하고, 다른 후보 모델에도 같은 평가 기준을 적용해 비교한다.

### 7. 다음 작업

- 휴관일 0을 평균에 포함·제외했을 때의 지표와 비용 차이 측정
- 팀 MLP 결과와 같은 조건으로 비교
- 최종 모델 선택 시 유형별 평가표·부족 상위 10건 재계산


In [43]:
mistake_rows = []

for origin in origins:
    for site, data in model_panel.groupby("site"):
        recent = data.loc[
            data["date"].ge(origin - pd.Timedelta(days=7))
            & data["date"].lt(origin)
        ]

        # 잘못된 방식: 휴관일 0 포함
        mean_with_zero = recent["total"].mean()

        # 현재 팀 방식: 휴관·결측 NaN 제외
        mean_open_only = recent["y"].mean()

        for date in pd.date_range(origin, periods=HORIZON):
            mistake_rows.append({
                "origin": origin,
                "site": site,
                "date": date,
                "mean_with_zero": mean_with_zero,
                "mean_open_only": mean_open_only
            })

mistake_predictions = pd.DataFrame(mistake_rows)

mistake_eval = mistake_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left",
    validate="many_to_one"
)

mistake_summary = pd.DataFrame([
    evaluate_predictions(
        mistake_eval,
        accuracy_col="mean_with_zero",
        staffing_col="mean_with_zero",
        name="휴관일 0 포함"
    ),
    evaluate_predictions(
        mistake_eval,
        accuracy_col="mean_open_only",
        staffing_col="mean_open_only",
        name="개방일만 평균"
    )
])

mistake_summary["앞 행 대비 절감액(원)"] = (
    mistake_summary["총비용(원)"].shift()
    - mistake_summary["총비용(원)"]
)

display(mistake_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),앞 행 대비 절감액(원)
0,휴관일 0 포함,1.154,43.393,47.994,1.524420e+09,1.970593e+08,1.721479e+09,NaN
1,개방일만 평균,1.218,44.417,58.189,1.191808e+09,2.938619e+08,1.485670e+09,2.358091e+08


## ⚠️ 실수 측정 1 · 휴관일 0을 평균에 포함하면?

### 비교 방법

직전 7일 평균을 두 방식으로 계산했다.

- **0 포함:** 원본 `total`의 휴관일 0까지 평균
- **개방일만:** 팀 전처리 `y`의 휴관·결측 NaN 제외

**평가 날짜와 MASE 분모는 동일하게 유지**해 평균 계산 방식의 차이를 비교했다. 서비스 수준 보정은 적용하지 않았다.

평가 범위는 **2025년 원점 52개·7일 예측, 1/1~12/30 중 실제 관람객이 있는 날**이다.

### 결과

| 항목 | 휴관일 0 포함 | 개방일만 평균 |
|---|---:|---:|
| MASE ↓ | **1.154** | 1.218 |
| WAPE ↓ | **43.393%** | 44.417% |
| 적중률 | 47.994% | **58.189%** |
| 부족 비용 | 약 15.24억 원 | 약 11.92억 원 |
| 과잉 비용 | 약 1.97억 원 | 약 2.94억 원 |
| 총비용 | 약 17.21억 원 | **약 14.86억 원** |

**개방일만 평균냈을 때 총비용 약 2억 3,581만 원, 13.7% 절감**

### 해석과 대응

휴관일 0을 포함하면 평균 예측과 배치 인력이 낮아진다. 이번 평가에서는 MASE·WAPE는 더 낮았지만, 인력 부족이 늘어 비용이 커졌다.

따라서 **휴관일 제외가 모든 정확도 지표를 개선한 것은 아니다.** 부족 비용이 과잉 비용보다 4배 큰 프로젝트 가정에서는 **개방일만 평균내는 방식이 배치 비용 측면에서 유리했다.**

**대응:** 원본 `total`은 보존하고, 과거 평균 계산에는 휴관·결측을 NaN으로 둔 `y`를 사용한다.

※ 이 절감액은 0 포함 방식과 개방일 평균의 비교 결과다. 모델 개선·서비스 수준 보정의 절감액에 별도로 더하지 않는다.

In [44]:
# 같은 평가 대상과 점예측을 사용
aggregation_check = notice_eval.loc[
    notice_eval["total"].gt(0)
].copy()

# 올바른 방식: 개소별 절대오차를 먼저 계산
site_absolute_error = (
    aggregation_check["point_pred"]
    - aggregation_check["total"]
).abs().sum()

# 잘못된 방식: 날짜별로 개소를 합친 뒤 절대오차 계산
daily_totals = aggregation_check.groupby(
    ["origin", "date"]
)[["total", "point_pred"]].sum()

aggregated_absolute_error = (
    daily_totals["point_pred"] - daily_totals["total"]
).abs().sum()

actual_total = aggregation_check["total"].sum()

aggregation_mistake_summary = pd.DataFrame([
    {
        "평가 방식": "개소별 오차 계산 후 합산",
        "절대오차 합(명)": site_absolute_error,
        "WAPE(%)": site_absolute_error / actual_total * 100
    },
    {
        "평가 방식": "개소 합산 후 오차 계산",
        "절대오차 합(명)": aggregated_absolute_error,
        "WAPE(%)": aggregated_absolute_error / actual_total * 100
    }
])

print(
    "합계 평가로 가려진 오차 비율(%):",
    round(
        (1 - aggregated_absolute_error / site_absolute_error) * 100,
        3
    )
)

display(aggregation_mistake_summary.round(3))

합계 평가로 가려진 오차 비율(%): 17.074


,평가 방식,절대오차 합(명),WAPE(%)
0,개소별 오차 계산 후 합산,5055196.436,28.456
1,개소 합산 후 오차 계산,4192047.892,23.597


## ⚠️ 실수 측정 2 · 개소를 합쳐서 채점하면?

**확인한 문제:** 한 개소의 과대예측과 다른 개소의 과소예측이 합계에서 서로 지워질 수 있다.

같은 날짜·예측값으로 두 계산 순서를 비교했다.

| 평가 방식 | WAPE |
|---|---:|
| 개소별 절대오차를 계산한 뒤 합산 | **28.456%** |
| 관람객·예측값을 먼저 합친 뒤 오차 계산 | 23.597% |

합계로 평가하면 **오차의 약 17.1%가 가려져**, WAPE가 **약 4.86%p 더 좋아 보였다.**

**대응:** 인력을 개소마다 배치하므로, 오차도 개소별로 먼저 계산한다.

※ WAPE는 이번 실수의 크기를 확인하는 데 사용했고, 주 평가 지표 MASE는 유지한다.

In [45]:
mistake_dir = Path("outputs/error_checks")
mistake_dir.mkdir(parents=True, exist_ok=True)

mistake_summary.to_csv(
    mistake_dir / "closed_day_zero_comparison.csv",
    index=False, encoding="utf-8-sig"
)

aggregation_mistake_summary.to_csv(
    mistake_dir / "site_aggregation_comparison.csv",
    index=False, encoding="utf-8-sig"
)

archive = shutil.make_archive(
    "error_checks_2025_52origins",
    "zip",
    root_dir=mistake_dir
)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 💾 실수 측정 결과 저장 완료

**저장 파일:** `error_checks_2025_52origins.zip`

가이드의 **‘자주 하는 실수 2가지 이상을 수치로 측정하고 대응 방법을 설명’**하는 항목을 위한 비교표다.

| 파일 | 확인한 내용 | 결과 |
|---|---|---|
| `closed_day_zero_comparison.csv` | 직전 7일 평균에 휴관일 0을 포함·제외했을 때의 차이 | 개방일만 평균내면 **비용 약 2억 3,581만 원 절감**. MASE·WAPE는 오히려 증가 |
| `site_aggregation_comparison.csv` | 개소별 오차 계산 vs 개소 합산 후 오차 계산 | 합계 평가에서 **오차 약 17.1%가 가려짐**. WAPE 28.456% → 23.597% |

**적용할 규칙**

- 과거 평균은 휴관·결측을 NaN으로 둔 `y`로 계산한다.
- 예측 오차는 **개소별로 먼저 계산**한다.
- 두 실험의 비교 대상이 다르므로 절감액·개선 효과를 합산하지 않는다.

**다음 작업:** 근무표를 **예측 기간 전날에 확정하는 경우와 1주 전에 확정하는 경우**를 같은 평가 날짜에서 비교한다. 확정 시점에 맞춰 과거 입력과 사전 공지 사용 가능 여부도 함께 제한한다.

In [46]:
def make_early_notice_inputs(origin):
    # 원점 7일 전에 근무표 확정
    cutoff = origin - pd.Timedelta(days=7)
    panel = free_panel.copy()

    # 확정 시점까지 공개된 공지만 반영
    known = notice_rules.loc[
        notice_rules["published"] <= cutoff
    ]

    for rule in known.itertuples():
        mask = panel["date"].between(rule.start, rule.end)

        if rule.site != "전체":
            mask &= panel["site"].eq(rule.site)

        panel.loc[mask, "is_free_rule"] = 1

    context = panel.loc[
        panel["date"] <= cutoff
    ].copy()

    # 평가 기간 이전 6일 + 평가할 7일 = 총 13일
    future = panel.loc[
        panel["date"].between(
            cutoff + pd.Timedelta(days=1),
            origin + pd.Timedelta(days=HORIZON - 1)
        ),
        ["site", "date", *free_features]
    ].copy()

    assert len(future) == 23 * 13
    return context, future


test_origin = origins[0]

early_context, early_future = make_early_notice_inputs(
    test_origin
)

early_raw = pipeline.predict_df(
    df=early_context,
    future_df=early_future,
    id_column="site",
    timestamp_column="date",
    target="y",
    prediction_length=13,
    quantile_levels=[0.5, 0.8],
    cross_learning=False,
    freq="D"
)

# 기존 모델과 같은 평가 기간의 7일만 남기기
early_test = early_raw.loc[
    pd.to_datetime(early_raw["date"]).between(
        test_origin,
        test_origin + pd.Timedelta(days=HORIZON - 1)
    )
].copy()

print("입력 마지막 날짜:", early_context["date"].max())
print("전체 예측 행 수:", len(early_raw))
print("평가 대상 행 수:", len(early_test))

입력 마지막 날짜: 2024-12-25 00:00:00
전체 예측 행 수: 299
평가 대상 행 수: 161


In [47]:
early_dir = Path("outputs/chronos2_early_schedule")
early_dir.mkdir(parents=True, exist_ok=True)

def format_early_result(raw, origin):
    # 13일 예측 중 기존 평가 날짜 7일만 선택
    selected = raw.loc[
        pd.to_datetime(raw["date"]).between(
            origin,
            origin + pd.Timedelta(days=HORIZON - 1)
        )
    ].copy()

    result = format_calendar_result(selected, origin)
    result["model"] = "chronos2_notice_schedule_7days_before"
    return result

# 첫 원점 시험 결과 재사용
format_early_result(early_raw, origins[0]).to_csv(
    early_dir / f"{origins[0]:%Y-%m-%d}.csv",
    index=False
)

early_results = []

for origin in tqdm(origins, desc="근무표 1주 전 확정"):
    checkpoint = early_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        context, future = make_early_notice_inputs(origin)

        raw = pipeline.predict_df(
            df=context,
            future_df=future,
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=13,
            quantile_levels=[0.5, 0.8],
            cross_learning=False,
            freq="D"
        )

        result = format_early_result(raw, origin)
        result.to_csv(checkpoint, index=False)

    early_results.append(result)

chronos_early_predictions = pd.concat(
    early_results, ignore_index=True
)

assert len(chronos_early_predictions) == 8372
assert not chronos_early_predictions.duplicated(
    ["origin", "site", "date"]
).any()

# 기존과 같은 실제값·평가 날짜·MASE 분모 사용
early_eval = chronos_early_predictions.merge(
    model_panel[["date", "site", "total"]],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left",
    validate="many_to_one"
)

early_summary = pd.DataFrame([
    evaluate_predictions(
        early_eval,
        accuracy_col="point_pred",
        staffing_col="q80",
        name="원점 7일 전 확정"
    )
])

late_summary = notice_summary.copy()
late_summary["방법"] = "원점 전날 확정"

schedule_comparison = pd.concat(
    [late_summary, early_summary],
    ignore_index=True
)

display(schedule_comparison.round(3))

print(
    "일찍 확정했을 때 총비용 변화(원):",
    round(
        early_summary.iloc[0]["총비용(원)"]
        - late_summary.iloc[0]["총비용(원)"]
    )
)

근무표 1주 전 확정:   0%|          | 0/52 [00:00<?, ?it/s]

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,원점 전날 확정,0.802,28.456,82.159,3.382963e+08,4.461166e+08,7.844128e+08
1,원점 7일 전 확정,0.870,29.199,81.992,3.283018e+08,4.693721e+08,7.976739e+08


일찍 확정했을 때 총비용 변화(원): 13261087


## 📅 근무표 확정 시점 비교

### 1. 무엇을 비교했나?

같은 공지 보정 Chronos 모델로, **근무표를 언제 확정하느냐에 따른 예측·배치 비용 차이**를 확인했다.

| 구분 | 과거 기록·공지 사용 범위 | 예측 방식 |
|---|---|---|
| 원점 전날 확정 | 원점 전날까지 | 다음 7일 예측 |
| 원점 7일 전 확정 | 원점 7일 전까지 | 13일 예측 후 앞 6일을 제외하고 같은 7일 평가 |

예를 들어 **2025년 1월 1~7일 근무표**를 만들 때, 전날 확정은 **12월 31일**, 7일 전 확정은 **12월 25일**까지의 정보만 사용한다.

**공통 조건:** 2025년 원점 52개·동일한 평가 날짜·0.8 분위수 배치·동일한 비용 가정. MASE 분모도 비교용으로 동일하게 고정했으며, 이른 시점의 예측 입력에는 사용하지 않았다.

### 2. 결과

| 항목 | 전날 확정 | 7일 전 확정 |
|---|---:|---:|
| MASE ↓ | **0.802** | 0.870 |
| WAPE ↓ | **28.456%** | 29.199% |
| 적중률 | 82.159% | 81.992% |
| 부족 비용 | 약 3.38억 원 | 약 3.28억 원 |
| 과잉 비용 | 약 4.46억 원 | 약 4.69억 원 |
| 총비용 | **약 7.84억 원** | 약 7.98억 원 |

**7일 전에 확정했을 때 총비용 약 1,326만 원, 1.7% 증가**

부족 비용은 약 999만 원 줄었지만, 과잉 비용이 약 2,326만 원 늘어 총비용이 증가했다.

### 3. 해석과 잠정 권고

- 이번 실험에서는 **전날 확정이 정확도와 배치 비용 측면에서 유리**했다.
- 7일 전 확정은 근무자가 일정을 **6일 더 일찍 알 수 있다는 장점**이 있다.
- 근무자의 일정 부담은 금액으로 계산하지 않았으므로, 추가 비용과 일정 안정성을 함께 고려해야 한다.
- **비용만 고려한 잠정 권고는 전날 확정**이며, 최종 모델이 바뀌면 다시 비교한다.

※ 두 방식 모두 **7일치 근무표를 한 번에 확정**한다. 매일 근무표를 갱신하는 방식과 비교한 것은 아니다.

**진행 상태:** 예측·평가 완료. 이번 비교 결과의 별도 파일 저장은 아직 진행 전이다.

In [48]:
schedule_dir = Path("outputs/schedule_timing")
schedule_dir.mkdir(parents=True, exist_ok=True)

chronos_early_predictions.to_csv(
    schedule_dir / "predictions_7days_before.csv",
    index=False,
    encoding="utf-8-sig"
)

chronos_notice_predictions.to_csv(
    schedule_dir / "predictions_day_before.csv",
    index=False,
    encoding="utf-8-sig"
)

schedule_comparison.to_csv(
    schedule_dir / "schedule_timing_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

archive = shutil.make_archive(
    "schedule_timing_2025_52origins",
    "zip",
    root_dir=schedule_dir
)

files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
print(chronos_notice_predictions.columns.tolist())

['origin', 'site', 'date', 'model', 'point_pred', 'q80']


In [51]:
q90_dir = Path("outputs/chronos2_notice_q90")
q90_dir.mkdir(parents=True, exist_ok=True)

q90_results = []

for origin in tqdm(origins, desc="0.9 분위수 예측"):
    checkpoint = q90_dir / f"{origin:%Y-%m-%d}.csv"

    if checkpoint.exists():
        result = pd.read_csv(
            checkpoint, parse_dates=["origin", "date"]
        )
    else:
        context, future = make_notice_inputs(origin)

        raw = pipeline.predict_df(
            df=context,
            future_df=future,
            id_column="site",
            timestamp_column="date",
            target="y",
            prediction_length=HORIZON,
            quantile_levels=[0.5, 0.8, 0.9],
            cross_learning=False,
            freq="D"
        )

        result = raw.rename(columns={
            "0.5": "point_pred",
            "0.8": "q80",
            "0.9": "q90"
        })[[
            "site", "date", "point_pred", "q80", "q90"
        ]].copy()

        result["date"] = pd.to_datetime(result["date"])
        result["origin"] = origin

        result[["point_pred", "q80", "q90"]] = (
            result[["point_pred", "q80", "q90"]].clip(lower=0)
        )

        assert len(result) == 23 * HORIZON
        assert not result.duplicated(["site", "date"]).any()
        assert np.isfinite(
            result[["point_pred", "q80", "q90"]].to_numpy()
        ).all()
        assert result["q90"].ge(result["q80"]).all()
        assert result["q80"].ge(result["point_pred"]).all()

        result.to_csv(checkpoint, index=False)

    q90_results.append(result)

notice_quantile_predictions = pd.concat(
    q90_results, ignore_index=True
)

assert len(notice_quantile_predictions) == 8372
assert not notice_quantile_predictions.duplicated(
    ["origin", "site", "date"]
).any()

# 다시 예측한 0.8 분위수가 기존 결과와 같은지 확인
check_q80 = notice_quantile_predictions.merge(
    chronos_notice_predictions[
        ["origin", "site", "date", "q80"]
    ],
    on=["origin", "site", "date"],
    validate="one_to_one",
    suffixes=("_new", "_old")
)

print("예측 행 수:", len(notice_quantile_predictions))
print(
    "기존 0.8 분위수와 차이의 최대값:",
    (
        check_q80["q80_new"] - check_q80["q80_old"]
    ).abs().max()
)

0.9 분위수 예측:   0%|          | 0/52 [00:00<?, ?it/s]

예측 행 수: 8372
기존 0.8 분위수와 차이의 최대값: 0.0037500000034924597


In [52]:
sensitivity_data = notice_quantile_predictions.merge(
    model_panel[["date", "site", "total", "is_free_rule"]],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
)

# 원점 확정 전에 공지된 무료일 포함
sensitivity_data["is_free_known"] = (
    sensitivity_data["is_free_rule"].eq(1)
)

for rule in notice_rules.itertuples():
    mask = (
        sensitivity_data["date"].between(rule.start, rule.end)
        & (
            sensitivity_data["origin"] - pd.Timedelta(days=1)
        ).ge(rule.published)
    )
    if rule.site != "전체":
        mask &= sensitivity_data["site"].eq(rule.site)

    sensitivity_data.loc[mask, "is_free_known"] = True

sensitivity_data = sensitivity_data.loc[
    sensitivity_data["total"].gt(0)
].copy()

def calculate_staffing(data, prediction, shortage_unit_cost, name):
    actual_staff = data["total"].astype("float64") / 2000
    assigned_staff = prediction.astype("float64") / 2000

    shortage = (actual_staff - assigned_staff).clip(lower=0)
    surplus = (assigned_staff - actual_staff).clip(lower=0)

    shortage_cost = (shortage * shortage_unit_cost).sum()
    surplus_cost = surplus.sum() * 150_000

    return {
        "설정": name,
        "평균 배치 인력": assigned_staff.mean(),
        "부족 인원 합": shortage.sum(),
        "과잉 인원 합": surplus.sum(),
        "적중률(%)": assigned_staff.ge(actual_staff).mean() * 100,
        "부족 비용(원)": shortage_cost,
        "과잉 비용(원)": surplus_cost,
        "총비용(원)": shortage_cost + surplus_cost
    }

# 무료일 부족 비용을 높인 가정
higher_cost = pd.Series(
    np.where(sensitivity_data["is_free_known"], 1_350_000, 600_000),
    index=sensitivity_data.index
)

mixed_prediction = sensitivity_data["q80"].where(
    ~sensitivity_data["is_free_known"],
    sensitivity_data["q90"]
)

sensitivity_summary = pd.DataFrame([
    calculate_staffing(
        sensitivity_data, sensitivity_data["q80"], 600_000,
        "기본 비용 · 모든 날 q80"
    ),
    calculate_staffing(
        sensitivity_data, sensitivity_data["q90"], 600_000,
        "기본 비용 · 모든 날 q90"
    ),
    calculate_staffing(
        sensitivity_data, sensitivity_data["q80"], higher_cost,
        "무료일 부족 비용 증가 · 기존 q80 유지"
    ),
    calculate_staffing(
        sensitivity_data, mixed_prediction, higher_cost,
        "무료일 부족 비용 증가 · 무료일 q90 적용"
    )
])

display(sensitivity_summary.round(3))

,설정,평균 배치 인력,부족 인원 합,과잉 인원 합,적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,기본 비용 · 모든 날 q80,1.573,563.827,2974.111,82.159,3.382963e+08,4.461166e+08,7.844128e+08
1,기본 비용 · 모든 날 q90,1.819,352.182,4528.558,91.852,2.113094e+08,6.792837e+08,8.905931e+08
2,무료일 부족 비용 증가 · 기존 q80 유지,1.573,563.827,2974.111,82.159,5.646985e+08,4.461166e+08,1.010815e+09
3,무료일 부족 비용 증가 · 무료일 q90 적용,1.645,483.594,3413.018,83.496,4.563837e+08,5.119527e+08,9.683364e+08


In [53]:
sensitivity_dir = Path("outputs/staffing_sensitivity")
sensitivity_dir.mkdir(parents=True, exist_ok=True)

sensitivity_summary.to_csv(
    sensitivity_dir / "sensitivity_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

notice_quantile_predictions.to_csv(
    sensitivity_dir / "predictions_q50_q80_q90.csv",
    index=False,
    encoding="utf-8-sig"
)

sensitivity_data[[
    "origin", "site", "date", "total",
    "q80", "q90", "is_free_known"
]].to_csv(
    sensitivity_dir / "evaluation_with_free_days.csv",
    index=False,
    encoding="utf-8-sig"
)

assumptions = {
    "visitors_per_staff": 2000,
    "surplus_cost_per_staff": 150000,
    "basic_shortage_cost_per_staff": 600000,
    "alternative_free_day_shortage_cost": 1350000,
    "basic_service_level": 0.8,
    "alternative_service_level": 0.9,
    "staffing_rounding": "비용 계산에서는 소수 인원 유지",
    "evaluation_rule": "실제 total > 0인 날",
    "free_day_rule": "명절·마지막 수요일 + 확정 전에 공지된 무료일",
    "note": "135만 원은 민감도 분석을 위한 가정"
}

with open(
    sensitivity_dir / "assumptions.json",
    "w", encoding="utf-8"
) as f:
    json.dump(assumptions, f, ensure_ascii=False, indent=2)

archive = shutil.make_archive(
    "staffing_sensitivity_2025_52origins",
    "zip",
    root_dir=sensitivity_dir
)

files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🔧 서비스 수준·비용 가정 변경 실험

### 1. 실험 목적

서비스 수준이나 인력 부족 비용을 바꾸면 **배치 인력·적중률·총비용이 얼마나 달라지는지** 확인했다.

**공통 조건:** 공지 보정 Chronos, 2025년 원점 52개·7일 예측, 실제 관람객이 있는 날 평가. 인력 1명당 관람객 2,000명이며, 비용은 올림 없이 소수 인원으로 계산했다.

### 2. 실험 ① 서비스 수준 0.8 → 0.9

부족 60만 원·과잉 15만 원의 **기본 비용은 유지**하고, 모든 날의 배치 기준만 바꿨다.

| 항목 | q80 배치 | q90 배치 |
|---|---:|---:|
| 평균 배치 인력 | 1.573명 | 1.819명 |
| 적중률 | 82.159% | **91.852%** |
| 부족 비용 | 약 3.38억 원 | 약 2.11억 원 |
| 과잉 비용 | 약 4.46억 원 | 약 6.79억 원 |
| 총비용 | **약 7.84억 원** | 약 8.91억 원 |

**q90 배치 시 총비용 약 1억 618만 원, 13.5% 증가**

부족은 줄었지만 과잉 비용이 더 크게 증가했다. 따라서 **이번 평가의 기본 비용에서는 q80 배치가 유리**했다.

### 3. 실험 ② 무료일의 부족 비용 증가

무료일에만 부족 비용을 **60만 → 135만 원**으로 높이고, 과잉 비용은 15만 원으로 유지했다.

135만 원은 실제 운영 비용이 아니라 **민감도 분석용 가정**이다. 이때 비용 기준 서비스 수준은 다음과 같다.

**135 ÷ (135 + 15) = 0.9**

같은 변경 비용 가정에서 두 배치 규칙을 비교했다.

| 항목 | 모든 날 q80 유지 | 무료일 q90·나머지 q80 |
|---|---:|---:|
| 평균 배치 인력 | 1.573명 | 1.645명 |
| 전체 적중률 | 82.159% | **83.496%** |
| 부족 비용 | 약 5.65억 원 | 약 4.56억 원 |
| 과잉 비용 | 약 4.46억 원 | 약 5.12억 원 |
| 총비용 | 약 10.11억 원 | **약 9.68억 원** |

**무료일만 q90 적용 시 총비용 약 4,248만 원, 4.2% 절감**

무료일 부족 손실을 더 크게 보는 가정에서는 **무료일에만 배치를 늘리는 규칙이 유리**했다.

### 4. 해석 시 주의

- 기본 비용에서는 **q80**, 무료일 부족 비용을 높인 가정에서는 **무료일 q90**이 유리했다.
- 비용 가정이 다른 실험끼리 총비용을 직접 비교하지 않는다.
- 평균 배치 인력은 **평가한 개소×날짜 한 건당 인원**이다.
- 부족·과잉 인원 합의 단위는 **인원·일**이며, 실제 총직원 수가 아니다.
- 이번에는 배치 기준을 바꾼 것으로, 점예측 정확도를 개선한 실험은 아니다.
- 무료일은 **명절·마지막 수요일 및 근무표 확정 전에 공지된 무료일**로 정의했다.

### 5. 저장 파일

**`staffing_sensitivity_2025_52origins.zip`**

| 파일 | 내용 |
|---|---|
| `sensitivity_summary.csv` | 가정별 배치 인력·적중률·비용 비교 |
| `predictions_q50_q80_q90.csv` | 모델의 0.5·0.8·0.9 분위수 예측 |
| `evaluation_with_free_days.csv` | 실제 관람객·배치용 예측·무료일 표시 |
| `assumptions.json` | 비용·서비스 수준·인원 계산 가정 |

**진행 상태:** 가정 변경 2가지 계산·저장 완료. 최종 모델이 바뀌면 같은 가정으로 다시 확인한다.